# Exact Minimum Vertex Cover (MVC) GPU Research: Automated Colab Ablation Suite
### Paper Title: *Accelerating Component-Aware GPU Branch-and-Bound for Minimum Vertex Cover via Lazy Splitting and Analytic Sub-Component Pruning*
**Domain:** Parallel Processing & GPU Computing (CUDA / C++)  
**Target:** 4-Variant Ablation Suite, Telemetry Extraction & Publication Plot Generation

---
### Four-Variant Ablation Matrix:
- **Variant A (2022 Baseline):** Component-oblivious GPU Branch-and-Reduce (Yamout et al., IEEE IPDPS 2022)
- **Variant B (2026 Baseline):** Always-on component-aware GPU Branch-and-Bound with Component Branch Registry (Amro et al., IEEE TPDS 2026)
- **Variant C (Addition 1 Only):** 2026 Baseline + Lazily Triggered Component Detection ($(\text{deg}(v) \ge \tau) \lor (d \pmod S == 0)$)
- **Variant D (Proposed Full Solver):** 2026 Baseline + Lazy Trigger + In-Block Analytic Tree & Cycle Fast-Path

---
> ⚠️ **IMPORTANT: Set Colab Runtime to GPU before running!**  
> Go to **Runtime** → **Change runtime type** → Select **T4 GPU** (or V100 / A100) → Click **Save**.


In [ ]:
# 1. Hardware Diagnostics & CUDA Architecture Detection
import os
import subprocess
import torch

print("="*70)
print("GPU HARDWARE & CUDA ENVIRONMENT DIAGNOSTICS")
print("="*70)

!nvidia-smi
!nvcc --version

# Auto-detect CUDA compute capability
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
print(f"\nDetected Active GPU: {gpu_name}")

compute_arch = "sm_75" # Default to T4 (sm_75)
if "T4" in gpu_name:
    compute_arch = "sm_75"
elif "V100" in gpu_name:
    compute_arch = "sm_70"
elif "A100" in gpu_name:
    compute_arch = "sm_80"
elif "A10" in gpu_name or "3090" in gpu_name or "3080" in gpu_name:
    compute_arch = "sm_86"
elif "H100" in gpu_name:
    compute_arch = "sm_90"

print(f"Optimal Compute Architecture: {compute_arch}")
print("="*70)


In [ ]:
# 2. Automated Workspace Setup & Codebase Deployment
# Clones both 2022 and 2026 foundational repositories and applies Variant C & Variant D optimizations
import shutil

WORKDIR = "/content/mvc_gpu_research"
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

print("Cloning foundational 2022 and 2026 repositories...")
if not os.path.exists("component-aware-vertex-cover-gpu"):
    !git clone https://github.com/HusseinAmro/component-aware-vertex-cover-gpu.git
if not os.path.exists("Parallel-Vertex-Cover-Algorithms-on-GPUs"):
    !git clone https://github.com/pky00/Parallel-Vertex-Cover-Algorithms-on-GPUs.git

print("Creating Variant C (Lazy Trigger) and Variant D (Full Solver) directories...")
if not os.path.exists("solver_variant_c_lazy"):
    shutil.copytree("component-aware-vertex-cover-gpu", "solver_variant_c_lazy")
if not os.path.exists("solver_variant_d_full"):
    shutil.copytree("component-aware-vertex-cover-gpu", "solver_variant_d_full")

# Inject optimized Variant C files
print("Deploying Variant C (Addition 1: Lazy Trigger) optimizations...")
with open("solver_variant_c_lazy/src/Counters.cuh", "w") as f:
    f.write('#ifndef COUNTERS_H\n#define COUNTERS_H\n\n#include <string.h>\n#include <sys/stat.h>\n#include "config.h"\n\nenum CounterName { ENQUEUE=0, DEQUEUE, FIND_BOUND, LEAF_REDUCTION, HIGH_DEGREE_REDUCTION, TRIANGLE_REDUCTION, COMP_SEARCH, MAX_DEGREE, NUM_EDGES, PREPARE_LEFT_CHILD, PREPARE_RIGHT_CHILD, PUSH_TO_STACK, POP_FROM_STACK, TERMINATE, MAX_DEPTH, TOTAL_BFS_SCANS, UNPRODUCTIVE_BFS_SCANS, CYCLE_FAST_PATH_HITS, TREE_FAST_PATH_HITS, NUM_COUNTERS };\n\nstruct Counters {\n    unsigned long long tmp[NUM_COUNTERS];\n    unsigned long long totalTime[NUM_COUNTERS];\n};\n\nstatic __device__ void initializeCounters(Counters* counters) {\n    #if USE_COUNTERS\n    if(threadIdx.x == 0) {\n        for(unsigned int i = 0; i < NUM_COUNTERS; ++i) {\n            counters->totalTime[i] = 0;\n        }\n    }\n    #endif\n}\n\nstatic __device__ void maxDepth(int stackTop, Counters* counters) {\n    #if USE_COUNTERS\n        if(threadIdx.x == 0) {\n            if(counters->totalTime[MAX_DEPTH] < stackTop){\n                counters->totalTime[MAX_DEPTH] = stackTop;\n            }\n        }\n    #endif\n}\n\nstatic __device__ void startTime(CounterName counterName, Counters* counters) {\n#if USE_COUNTERS\n    if(threadIdx.x == 0) {\n        counters->tmp[counterName] = clock64();\n    }\n#endif\n}\n\nstatic __device__ void endTime(CounterName counterName, Counters* counters) {\n#if USE_COUNTERS\n    if(threadIdx.x == 0) {\n        counters->totalTime[counterName] += clock64() - counters->tmp[counterName];\n    }\n#endif\n}\n\nvoid printCountersInFile(Config config, Counters * counters_d, unsigned int numBlocks, double reduceAndInduceTime, double kernelTime){\n\n    Counters * counters;\n    counters = (Counters *)malloc(sizeof(Counters)*numBlocks);\n    cudaMemcpy(counters, counters_d, sizeof(Counters)*numBlocks, cudaMemcpyDeviceToHost);\n\n    char filename[100];\n    strcpy(filename,config.graphFileName);\n\n    char * token = strtok(filename, "/");\n    // loop through the string to extract all other tokens\n    char * outputfile;\n    while( token != NULL ) {\n        outputfile = token;\n        token = strtok(NULL, "/");\n    }\n\n    char outputFilename[500];\n    strcpy(outputFilename,"Counters/Counters_");\n    strcat(outputFilename,outputfile);\n\n    mkdir("Counters", 0777);\n    FILE *output_file = fopen(outputFilename, "w"); \n\n    fprintf(output_file,"BLOCK_NO,Add to worklist,Remove from worklist,Find boundaries,Degree-one rule,High-degree rule,Degree-two-triangle rule,Components search,Find max degree vertex,Stopping condition,Remove max-degree vertex,Remove neighbors of max-degree vertex,Push to stack,Pop from stack,Terminate,MAX_DEPTH,Total BFS Scans,Unproductive BFS Scans,Cycle Fast-Path Hits,Tree Fast-Path Hits\\n");\n \n    for(unsigned int i = 0;i<numBlocks;++i){\n        fprintf(output_file,"%u",i);\n        for(unsigned int j = 0;j<NUM_COUNTERS;++j){\n            fprintf(output_file,",%llu",counters[i].totalTime[j]);\n        }\n        fprintf(output_file,"\\n");\n    }\n\n    fprintf(output_file, "\\n");\n    fprintf(output_file, "Reduce and Induce time: %fms\\n", reduceAndInduceTime);\n    fprintf(output_file, "Kernel time: %fms\\n", kernelTime);\n\n    free(counters);\n    fclose(output_file);\n}\n\n__device__ int get_smid(void) {\n    uint ret;\n    asm("mov.u32 %0, %smid;" : "=r"(ret) );\n    return ret;\n}\n\nvoid printNodesPerSM(Config config, unsigned long long * NODES_PER_SM,  int numOfSMs){\n\n    unsigned long long * NODES_PER_SM_host;\n    NODES_PER_SM_host = (unsigned long long *)malloc(sizeof(unsigned long long)*numOfSMs);\n    cudaMemcpy(NODES_PER_SM_host, NODES_PER_SM, sizeof(unsigned long long)*numOfSMs, cudaMemcpyDeviceToHost);\n\n    char filename[100];\n    strcpy(filename,config.graphFileName);\n\n    char * token = strtok(filename, "/");\n    // loop through the string to extract all other tokens\n    char * outputfile;\n    while( token != NULL ) {\n        outputfile = token;\n        token = strtok(NULL, "/");\n    }\n\n    char outputFilename[500];\n    strcpy(outputFilename,"NODES_PER_SM/NODES_PER_SM_");\n    strcat(outputFilename,outputfile);\n\n    mkdir("NODES_PER_SM", 0777);\n    FILE *output_file = fopen(outputFilename, "w");\n\n\tfor(unsigned int i = 0;i<numOfSMs;++i){\n\t\tchar str[100] = "SM_";\n\t\tfprintf(output_file,"SM_%u",i+1);\n\t\tif (i < numOfSMs - 1){\n\t\t\tfprintf(output_file,",");\n\t\t}\n\t}\n\tfprintf(output_file,"\\n");\n \n    for(unsigned int i = 0;i<numOfSMs;++i){\n\t\tfprintf(output_file,"%llu",NODES_PER_SM_host[i]);\n\t\tif (i < numOfSMs - 1){\n\t\t\tfprintf(output_file,",");\n\t\t}\n    }\n\n    free(NODES_PER_SM_host);\n    fclose(output_file);\n}\n\n\n#endif\n')

with open("solver_variant_c_lazy/src/GlobalWorkList.cuh", "w") as f:
    f.write('#include <stdio.h>\n#include <stdint.h>\n#include <assert.h>\n\n#include "config.h"\n#include "stack.cuh"\n#include "Counters.cuh"\n#include "BWDWorkList.cuh"\n#include "helperFunctions.cuh"\n\n#ifndef LAZY_TAU\n#define LAZY_TAU 4        // Hub-removal degree threshold (tau)\n#endif\n\n#ifndef LAZY_STRIDE\n#define LAZY_STRIDE 2     // Search-tree depth stride (S)\n#endif\n\n#if USE_SHORT_DEGREE\n#define INT_OR_SHORT short\n#define FUNCTION_NAME(base) base##_short\n#define SHARED_MEM_TYPE(base) base##_short\n#else\n#define INT_OR_SHORT int\n#define FUNCTION_NAME(base) base##_int\n#define SHARED_MEM_TYPE(base) base##_int\n#endif\n\n#if USE_GLOBAL_MEMORY\n__global__ void FUNCTION_NAME(GlobalWorkList_global_kernel)(Stacks<INT_OR_SHORT> stacks, unsigned int * minimum, WorkList<INT_OR_SHORT> workList, CSRGraph graph, Counters* counters, int* first_to_dequeue_global, INT_OR_SHORT* global_memory, unsigned long long* NODES_PER_SM, CompArray compArray, unsigned int* compArraySize, unsigned int findBound, unsigned int* numBranches, unsigned int* numBranchesFreq) {\n#else\n__global__ void FUNCTION_NAME(GlobalWorkList_shared_kernel)(Stacks<INT_OR_SHORT> stacks, unsigned int * minimum, WorkList<INT_OR_SHORT> workList, CSRGraph graph, Counters* counters, int* first_to_dequeue_global, unsigned long long* NODES_PER_SM, CompArray compArray, unsigned int* compArraySize, unsigned int findBound, unsigned int* numBranches, unsigned int* numBranchesFreq) {\n#endif\n\n    // For Evaluation Purposes\n    //__shared__ Counters blockCounters;\n \n    Counters blockCounters;\n    initializeCounters(&blockCounters);\n    \n    // For Evaluation Purposes\n    #if RUN_KILL_TIMEOUT\n    __shared__ long long startingTime;\n    __shared__ bool timeOut;\n    if (threadIdx.x==0) {\n        startingTime = clock64();\n        timeOut = false;\n    }\n    #endif\n\n    #if USE_COUNTERS\n        __shared__ unsigned int sm_id;\n        if (threadIdx.x==0){\n            sm_id=get_smid();\n        }\n    #endif\n    int stackTop = -1;\n    unsigned int stackSize = (stacks.minimum + 1);\n    volatile INT_OR_SHORT * stackVertexDegrees = &stacks.stacks[(long long)blockIdx.x * (long long)stackSize * (long long)graph.vertexNum];\n    volatile unsigned int * stackNumDeletedVertices = &stacks.stacksNumDeletedVertices[blockIdx.x * stackSize];\n    volatile int * stackArrayIndex = &stacks.stackArrayIndex[blockIdx.x * stackSize];\n\n    __shared__ int s_current_depth;\n    __shared__ int s_last_branched_degree;\n    __shared__ bool run_cc_scan;\n\n    // Define the vertexDegree_s\n    unsigned int numDeletedVertices;\n    unsigned int numDeletedVertices2;\n    \n    int arrayIndex = -1;\n    \n    #if USE_GLOBAL_MEMORY\n    INT_OR_SHORT * vertexDegrees_s = &global_memory[graph.vertexNum*(2*blockIdx.x)];\n    INT_OR_SHORT * vertexDegrees_s2 = &global_memory[graph.vertexNum*(2*blockIdx.x + 1)];\n    #else\n    extern __shared__ INT_OR_SHORT SHARED_MEM_TYPE(shared_mem)[];\n    INT_OR_SHORT * vertexDegrees_s = SHARED_MEM_TYPE(shared_mem);\n    INT_OR_SHORT * vertexDegrees_s2 = &SHARED_MEM_TYPE(shared_mem)[graph.vertexNum];\n    #endif\n    \n    bool dequeueOrPopNextItr = true; \n    __syncthreads();\n\n    __shared__ bool first_to_dequeue;\n    if (threadIdx.x==0) {\n        if(atomicCAS(first_to_dequeue_global,0,1) == 0) { \n            first_to_dequeue = true;\n        } else {\n            first_to_dequeue = false;\n        }\n    }\n    __syncthreads();\n    if (first_to_dequeue) {\n        for(unsigned int vertex = threadIdx.x; vertex < graph.vertexNum; vertex += blockDim.x) {\n            vertexDegrees_s[vertex] = graph.degree[vertex];\n        }\n        numDeletedVertices = 0;\n        dequeueOrPopNextItr = false;\n        if (threadIdx.x == 0) {\n            s_current_depth = 0;\n            s_last_branched_degree = 9999;\n        }\n        #if USE_COUNTERS\n            if (threadIdx.x==0){\n                atomicAdd(&NODES_PER_SM[sm_id],1);\n            }\n        #endif\n    }\n    __syncthreads();\n\n    while(true) {\n        if(dequeueOrPopNextItr) {\n            if(stackTop != -1) { // Local stack is not empty, pop from the local stack\n                startTime(POP_FROM_STACK,&blockCounters);\n                popStack(graph.vertexNum, vertexDegrees_s, &numDeletedVertices, &arrayIndex, NULL, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, NULL, &stackTop, false);\n                endTime(POP_FROM_STACK,&blockCounters);\n\n                if (threadIdx.x == 0) {\n                    s_current_depth = stackTop + 1;\n                    s_last_branched_degree = 9999;\n                }\n\n                #if USE_COUNTERS\n                    if (threadIdx.x==0){\n                        atomicAdd(&NODES_PER_SM[sm_id],1);\n                    }\n                #endif\n            } else { // Local stack is empty, read from the global workList\n                startTime(TERMINATE,&blockCounters);\n                startTime(DEQUEUE,&blockCounters);\n                if(!dequeue(vertexDegrees_s, workList, graph.vertexNum, &numDeletedVertices, &arrayIndex)) {   \n                    endTime(TERMINATE,&blockCounters);\n                    break;\n                }\n                endTime(DEQUEUE,&blockCounters);\n\n                if (threadIdx.x == 0) {\n                    s_current_depth = 0;\n                    s_last_branched_degree = 9999;\n                }\n                \n                // For Debugging Purposes\n                /*__shared__ bool conditionn;\n                if (threadIdx.x==0) {\n                    conditionn = false;\n                    for (int i = 0; i<graph.vertexNum; ++i) {\n                        if (vertexDegrees_s[i]>0) {\n                            unsigned int counter = 0;\n                            for(unsigned int edge = graph.srcPtr[i]; edge<graph.srcPtr[i+1]; ++edge) {\n                                unsigned int neigh = graph.dst[edge];\n                                if(vertexDegrees_s[neigh]>0) {\n                                    ++counter;\n                                }\n                            }\n                            if (counter != vertexDegrees_s[i]) {\n                                conditionn = true;\n                            }\n                        }\n                    }\n                    if (conditionn) {\n                        printf("Head = %u, Tail = %u, woklist Baaaaaaaaaaaaaad count = %d, numWait = %d, numEnq = %d\\n",\n                           (unsigned int)(*workList.head_tail & 0xFFFFFFFF),\n                           (unsigned int)((*workList.head_tail >> 32) & 0xFFFFFFFF),\n                           *workList.count,\n                           workList.counter->numWaiting,\n                           workList.counter->numEnqueued\n                        );\n                    } else {\n                        printf("Head = %u, Tail = %u, woklist count = %d, numWait = %d, numEnq = %d\\n",\n                           (unsigned int)(*workList.head_tail & 0xFFFFFFFF),\n                           (unsigned int)((*workList.head_tail >> 32) & 0xFFFFFFFF),\n                           *workList.count,\n                           workList.counter->numWaiting,\n                           workList.counter->numEnqueued\n                        );\n                    }\n                }\n                __syncthreads();*/\n\n                #if USE_COUNTERS\n                    if (threadIdx.x==0){\n                        atomicAdd(&NODES_PER_SM[sm_id],1);\n                    }\n                #endif\n            }\n        }\n        __syncthreads();\n        \n        __shared__ unsigned int left;\n        __shared__ unsigned int right;\n        if(threadIdx.x == 0){\n            left = graph.vertexNum-1;\n            right = 0;\n        }\n        __syncthreads();\n        \n        if (findBound) {\n            startTime(FIND_BOUND,&blockCounters);\n            findBoundaries(vertexDegrees_s, &left, &right, graph.vertexNum);\n            endTime(FIND_BOUND,&blockCounters);\n        }\n\n        __shared__ unsigned int minimum_s;\n        if(threadIdx.x == 0) {\n            if (arrayIndex>-1) {\n                minimum_s = atomicOr((int*)&compArray.entries[arrayIndex].minimum,0);\n            } else {\n                minimum_s = atomicOr(minimum,0);\n            }\n            \n            if (right == 0) {\n                left = 0;\n                right = graph.vertexNum-1;\n            }\n            \n            // For Evaluation Purposes\n            #if RUN_KILL_TIMEOUT\n            if (clock64() - startingTime > 25000000000000) {\n                timeOut = true;\n            }\n            #endif\n        }\n        __syncthreads();\n\n        // For Evaluation Purposes\n        #if RUN_KILL_TIMEOUT\n        if (timeOut) {\n            stackTop = -1;\n            dequeueOrPopNextItr = true;\n            continue;\n        }\n        #endif\n\n        unsigned int iterationCounter = 0, numDeletedVerticesLeaf, numDeletedVerticesTriangle, numDeletedVerticesHighDegree;\n        do {\n            startTime(LEAF_REDUCTION,&blockCounters);\n            numDeletedVerticesLeaf = leafReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, left, right);\n            endTime(LEAF_REDUCTION,&blockCounters);\n            numDeletedVertices += numDeletedVerticesLeaf;\n            if(iterationCounter==0 || numDeletedVerticesLeaf>0 || numDeletedVerticesHighDegree>0) {\n                startTime(TRIANGLE_REDUCTION,&blockCounters);\n                numDeletedVerticesTriangle = triangleReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, left, right);\n                endTime(TRIANGLE_REDUCTION,&blockCounters);\n                numDeletedVertices += numDeletedVerticesTriangle;\n            } else {\n                numDeletedVerticesTriangle = 0;\n            }\n            if(iterationCounter==0 || numDeletedVerticesLeaf>0 || numDeletedVerticesTriangle>0) {\n                startTime(HIGH_DEGREE_REDUCTION,&blockCounters);\n                numDeletedVerticesHighDegree = highDegreeReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2,numDeletedVertices,minimum_s, left, right);\n                endTime(HIGH_DEGREE_REDUCTION,&blockCounters);\n                numDeletedVertices += numDeletedVerticesHighDegree;\n            } else {\n                numDeletedVerticesHighDegree = 0;\n            }\n            ++iterationCounter;\n        } while(numDeletedVerticesTriangle>0 || numDeletedVerticesHighDegree>0);\n        \n        startTime(NUM_EDGES,&blockCounters);\n        unsigned int numOfEdges = findNumOfEdges(graph.vertexNum, vertexDegrees_s, vertexDegrees_s2, left, right);\n        endTime(NUM_EDGES,&blockCounters);\n\n        bool numEdgesCondition = false;\n        if(numOfEdges>=square((int)minimum_s-(int)numDeletedVertices-1)+1) {\n            numEdgesCondition = true;\n        }\n        __syncthreads();\n\n        if(threadIdx.x == 0) {\n            if (arrayIndex>-1) {\n                minimum_s = atomicOr((int*)&compArray.entries[arrayIndex].minimum,0);\n            } else {\n                minimum_s = atomicOr(minimum,0);\n            }\n            // For Debugging Purposes\n            //printf("I am Alive blockIdx.x = %d and stackTop = %d and minim = %u\\n", blockIdx.x, stackTop, minimum_s);\n        }\n        __syncthreads();\n\n        if(numDeletedVertices>=minimum_s || numEdgesCondition) { // Reached the bottom of the tree, no minimum vertex cover found\n            if(threadIdx.x==0 && arrayIndex>-1) {\n                int counter = atomicSub((int*)&compArray.entries[arrayIndex].counter, 1);\n                if (counter==1) {\n                    int parentIndex = compArray.entries[arrayIndex].index;\n                    atomicAdd((int*)&compArray.entries[parentIndex].minimum, compArray.entries[arrayIndex].minimum);\n                    int parentCounter = atomicSub((int*)&compArray.entries[parentIndex].counter, 1);\n                    if (parentCounter==1) {\n                        arrayIndex = compArray.entries[parentIndex].index;\n                        numDeletedVertices = atomicOr((int*)&compArray.entries[parentIndex].minimum,0);\n                        while (true) {\n                            if (arrayIndex>-1) {\n                                atomicMin((int*)&compArray.entries[arrayIndex].minimum, numDeletedVertices);\n                                int counter = atomicSub((int*)&compArray.entries[arrayIndex].counter, 1);\n                                if (counter-1>0) {\n                                    break;\n                                }\n                                int parentIndex = compArray.entries[arrayIndex].index;\n                                atomicAdd((int*)&compArray.entries[parentIndex].minimum, compArray.entries[arrayIndex].minimum);\n                                int parentCounter = atomicSub((int*)&compArray.entries[parentIndex].counter, 1);\n                                if (parentCounter-1>0) {\n                                    break;\n                                }\n                                arrayIndex = compArray.entries[parentIndex].index;\n                                numDeletedVertices = atomicOr((int*)&compArray.entries[parentIndex].minimum,0);\n\n                            } else {\n                                atomicMin(minimum, numDeletedVertices);\n                                break;\n                            }\n                        }\n                    }\n                }\n            }\n            dequeueOrPopNextItr = true;\n            __syncthreads();\n\n        } else {\n\n            if (threadIdx.x == 0) {\n                run_cc_scan = (s_current_depth == 0) || \n                              (s_last_branched_degree >= LAZY_TAU) || \n                              (s_current_depth % LAZY_STRIDE == 0);\n            }\n            __syncthreads();\n\n            int isComponents = 1;\n            if (run_cc_scan) {\n                #if USE_COUNTERS\n                if (threadIdx.x == 0) {\n                    blockCounters.totalTime[TOTAL_BFS_SCANS]++;\n                }\n                #endif\n                startTime(COMP_SEARCH,&blockCounters);\n                isComponents = bfsCC_Hybrid(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, arrayIndex, minimum_s, &numDeletedVertices, compArray, workList, &blockCounters, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, &stackTop, stackSize, *compArraySize, left, right, minimum);\n                endTime(COMP_SEARCH,&blockCounters);\n                #if USE_COUNTERS\n                if (threadIdx.x == 0 && isComponents == 1) {\n                    blockCounters.totalTime[UNPRODUCTIVE_BFS_SCANS]++;\n                }\n                #endif\n            } else {\n                isComponents = 1;\n            }\n            __syncthreads();\n\n            unsigned int maxVertex;\n            INT_OR_SHORT maxDegree;\n            if (isComponents>1) {\n                #if COUNT_BRANCHING_ON_COMPONENTS\n                if (threadIdx.x==0) {\n                   atomicAdd(numBranches, 1);\n                   atomicAdd(&numBranchesFreq[isComponents], 1);\n                }\n                __syncthreads();\n                #endif\n                dequeueOrPopNextItr = true;\n                __syncthreads();\n\n            } else if (isComponents==0) { // Reached the bottom of the tree, minimum vertex cover possibly found\n                if(threadIdx.x==0) {\n                    while (true) {\n                        if (arrayIndex>-1) {\n                            atomicMin((int*)&compArray.entries[arrayIndex].minimum, numDeletedVertices);\n                            int counter = atomicSub((int*)&compArray.entries[arrayIndex].counter, 1);\n                            if (counter-1>0) {\n                                break;\n                            }\n                            int parentIndex = compArray.entries[arrayIndex].index;\n                            atomicAdd((int*)&compArray.entries[parentIndex].minimum, compArray.entries[arrayIndex].minimum);\n                            int parentCounter = atomicSub((int*)&compArray.entries[parentIndex].counter, 1);\n                            if (parentCounter-1>0) {\n                                break;\n                            }\n                            arrayIndex = compArray.entries[parentIndex].index;\n                            numDeletedVertices = atomicOr((int*)&compArray.entries[parentIndex].minimum,0);\n                        } else {\n                            atomicMin(minimum, numDeletedVertices);\n                            break;\n                        }\n                    }\n                }\n                dequeueOrPopNextItr = true;\n                __syncthreads();\n\n            } else {\n                // Vertex cover not found, need to branch\n                if(threadIdx.x == 0) {\n                    if (arrayIndex>-1) {\n                        atomicAdd((int*)&compArray.entries[arrayIndex].counter, 1);\n                    }\n                }\n                __syncthreads();\n                \n                startTime(MAX_DEGREE,&blockCounters);\n                findMaxDegree(graph.vertexNum, &maxVertex, &maxDegree, vertexDegrees_s, vertexDegrees_s2, left, right);\n                endTime(MAX_DEGREE,&blockCounters);\n\n                startTime(PREPARE_RIGHT_CHILD,&blockCounters);\n                deleteNeighborsOfMaxDegreeVertex(graph,vertexDegrees_s, &numDeletedVertices, vertexDegrees_s2, &numDeletedVertices2, maxDegree, maxVertex);\n                endTime(PREPARE_RIGHT_CHILD,&blockCounters);\n\n                bool enqueueSuccess;\n                if(checkThreshold(workList)) {\n                    startTime(ENQUEUE,&blockCounters);\n                    enqueueSuccess = enqueue(vertexDegrees_s2, workList, graph.vertexNum, &numDeletedVertices2, arrayIndex);\n                } else  {\n                    enqueueSuccess = false;\n                }\n                __syncthreads();\n\n                if(!enqueueSuccess) {\n                    startTime(PUSH_TO_STACK,&blockCounters);\n                    pushStack(graph.vertexNum, vertexDegrees_s2, &numDeletedVertices2, arrayIndex, NULL, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, NULL, &stackTop, false);\n                    endTime(PUSH_TO_STACK,&blockCounters);\n                    maxDepth(stackTop, &blockCounters);\n                } else {\n                    endTime(ENQUEUE,&blockCounters);\n                }\n                __syncthreads();\n\n                startTime(PREPARE_LEFT_CHILD,&blockCounters);\n                // Prepare the child that removes the neighbors of the max vertex to be processed on the next iteration\n                deleteMaxDegreeVertex(graph, vertexDegrees_s, &numDeletedVertices, maxVertex);\n                endTime(PREPARE_LEFT_CHILD,&blockCounters);\n\n                #if USE_COUNTERS\n                    if (threadIdx.x==0) {\n                        atomicAdd(&NODES_PER_SM[sm_id],1);\n                    }\n                #endif\n\n                dequeueOrPopNextItr = false;\n                if (threadIdx.x == 0) {\n                    s_current_depth++;\n                    s_last_branched_degree = maxDegree;\n                }\n                __syncthreads();\n            }\n        }\n        __syncthreads();\n    }\n\n    // For Debugging Purposes\n    /*if (blockIdx.x==0) {\n        if (threadIdx.x==0) {\n            int globIndex = (*compArray.globalArrayIndex<*compArraySize)?*compArray.globalArrayIndex:*compArraySize;\n            for(unsigned int vertex = 0; vertex<globIndex; vertex++) {\n                printf("index = %d\\n",vertex);\n                printf("counter = %d\\n",compArray.entries[vertex].counter);\n                printf("minimum = %d\\n",compArray.entries[vertex].minimum);\n                printf("index = %d\\n\\n",compArray.entries[vertex].index);\n            }\n        }\n    }*/\n\n    #if USE_COUNTERS\n    if(threadIdx.x == 0) {\n        counters[blockIdx.x] = blockCounters;\n    }\n    #endif\n}\n\n#undef INT_OR_SHORT\n#undef FUNCTION_NAME(base)\n#undef SHARED_MEM_TYPE(base)')

with open("solver_variant_c_lazy/src/LocalStacks.cuh", "w") as f:
    f.write('#include <stdio.h>\n#include <stdint.h>\n#include <assert.h>\n\n#include "config.h"\n#include "stack.cuh"\n#include "Counters.cuh"\n#include "helperFunctions.cuh"\n\n#ifndef LAZY_TAU\n#define LAZY_TAU 4        // Hub-removal degree threshold (tau)\n#endif\n\n#ifndef LAZY_STRIDE\n#define LAZY_STRIDE 2     // Search-tree depth stride (S)\n#endif\n\n#if USE_SHORT_DEGREE\n#define INT_OR_SHORT short\n#define FUNCTION_NAME(base) base##_short\n#define SHARED_MEM_TYPE(base) base##_short\n#else\n#define INT_OR_SHORT int\n#define FUNCTION_NAME(base) base##_int\n#define SHARED_MEM_TYPE(base) base##_int\n#endif\n\n#if USE_GLOBAL_MEMORY\n__global__ void FUNCTION_NAME(LocalStacks_global_kernel)(Stacks<INT_OR_SHORT> stacks, CSRGraph graph, unsigned int * minimum, INT_OR_SHORT* global_memory, Counters* counters, unsigned int * pathCounter, unsigned long long* NODES_PER_SM, int startingDepth, unsigned int findBound) {\n#else\n__global__ void FUNCTION_NAME(LocalStacks_shared_kernel)(Stacks<INT_OR_SHORT> stacks, CSRGraph graph, unsigned int * minimum, Counters* counters, unsigned int * pathCounter, unsigned long long* NODES_PER_SM, int startingDepth, unsigned int findBound) {\n#endif\n\n    Counters blockCounters;\n    initializeCounters(&blockCounters);\n    \n    __shared__ unsigned int left;\n    __shared__ unsigned int right;\n    __shared__ int s_current_depth;\n    __shared__ int s_last_branched_degree;\n    __shared__ bool run_cc_scan;\n\n    #if USE_COUNTERS\n        __shared__ unsigned int sm_id;\n        if (threadIdx.x==0){\n            sm_id=get_smid();\n        }\n    #endif\n\n    do{\n        __shared__ unsigned int path;\n        if(threadIdx.x==0){\n            path = atomicAdd(pathCounter,1);\n        }\n        __syncthreads();\n\n        if(path>=(1<<startingDepth)){\n            break;\n        }\n\n        // Initialize the vertexDegrees_s\n        \n        unsigned int numDeletedVertices = 0;\n        unsigned int numDeletedVertices2;\n\n        #if USE_GLOBAL_MEMORY\n        INT_OR_SHORT * vertexDegrees_s = &global_memory[graph.vertexNum*(2*blockIdx.x)];\n        INT_OR_SHORT * vertexDegrees_s2 = &global_memory[graph.vertexNum*(2*blockIdx.x + 1)];\n        #else\n        extern __shared__ INT_OR_SHORT SHARED_MEM_TYPE(shared_mem)[];\n        INT_OR_SHORT * vertexDegrees_s = SHARED_MEM_TYPE(shared_mem);\n        INT_OR_SHORT * vertexDegrees_s2 = &SHARED_MEM_TYPE(shared_mem)[graph.vertexNum];\n        #endif\n\n        for(unsigned int i = threadIdx.x; i < graph.vertexNum; i += blockDim.x) {\n            vertexDegrees_s[i] = graph.degree[i];\n        }\n        __syncthreads();\n\n        // Find the block\'s sub-tree\n        bool vcFound = false;\n        bool minExceeded = false;\n        \n        for(unsigned int k = 0; k < startingDepth && !vcFound && !minExceeded; ++k) {\n            #if USE_COUNTERS\n                if (threadIdx.x==0){\n                    atomicAdd(&NODES_PER_SM[sm_id],1);\n                }\n            #endif\n            \n            if(threadIdx.x == 0){\n                left = graph.vertexNum-1;\n                right = 0;\n            }\n            __syncthreads();\n            if (k > 0) {\n                if (findBound) {\n                    startTime(FIND_BOUND,&blockCounters);\n                    findBoundaries(vertexDegrees_s, &left, &right, graph.vertexNum);\n                    endTime(FIND_BOUND,&blockCounters);\n                }\n            } else {\n                if(threadIdx.x == 0) {\n                    left = 0;\n                    right = graph.vertexNum -1;\n                }\n            }\n            __syncthreads();\n\n            // reduction rule\n            __shared__ unsigned int minimum_s;\n\n            if(threadIdx.x == 0) {\n                minimum_s = atomicOr(minimum,0);\n                if (right == 0) {\n                    left = 0;\n                    right = graph.vertexNum-1;\n                }\n            }\n            __syncthreads();\n\n            unsigned int iterationCounter = 0, numDeletedVerticesLeaf, numDeletedVerticesTriangle, numDeletedVerticesHighDegree;\n            do {\n                startTime(LEAF_REDUCTION,&blockCounters);\n                numDeletedVerticesLeaf = leafReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, left, right);\n                endTime(LEAF_REDUCTION,&blockCounters);\n                numDeletedVertices += numDeletedVerticesLeaf;\n                if(iterationCounter==0 || numDeletedVerticesLeaf>0 || numDeletedVerticesHighDegree>0) {\n                    startTime(TRIANGLE_REDUCTION,&blockCounters);\n                    numDeletedVerticesTriangle = triangleReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, left, right);\n                    endTime(TRIANGLE_REDUCTION,&blockCounters);\n                    numDeletedVertices += numDeletedVerticesTriangle;\n                } else {\n                    numDeletedVerticesTriangle = 0;\n                }\n                if(iterationCounter==0 || numDeletedVerticesLeaf>0 || numDeletedVerticesTriangle>0) {\n                    startTime(HIGH_DEGREE_REDUCTION,&blockCounters);\n                    numDeletedVerticesHighDegree = highDegreeReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2,numDeletedVertices,minimum_s, left, right);\n                    endTime(HIGH_DEGREE_REDUCTION,&blockCounters);\n                    numDeletedVertices += numDeletedVerticesHighDegree;\n                } else {\n                    numDeletedVerticesHighDegree = 0;\n                }\n                ++iterationCounter;\n            } while(numDeletedVerticesTriangle>0 || numDeletedVerticesHighDegree>0);\n\n            startTime(NUM_EDGES,&blockCounters);\n            unsigned int numOfEdges = findNumOfEdges(graph.vertexNum, vertexDegrees_s, vertexDegrees_s2, left, right);\n            endTime(NUM_EDGES,&blockCounters);\n            \n            bool numEdgesCondition = false;\n            if(numOfEdges>=square((int)minimum_s-(int)numDeletedVertices-1)+1) {\n                numEdgesCondition = true;\n            }\n            __syncthreads();\n\n            if(threadIdx.x == 0) {\n                minimum_s = atomicOr(minimum,0);\n            }\n            __syncthreads();\n            \n            if(numDeletedVertices >= minimum_s || numEdgesCondition) {\n                minExceeded = true;\n            } else {\n\n                unsigned int maxVertex = 0;\n                INT_OR_SHORT maxDegree = 0;\n                \n                startTime(MAX_DEGREE,&blockCounters);\n                findMaxDegree(graph.vertexNum, &maxVertex, &maxDegree, vertexDegrees_s, vertexDegrees_s2, left, right);\n                endTime(MAX_DEGREE,&blockCounters);\n\n                if(maxDegree<1){\n                    vcFound = true;\n                    if(threadIdx.x == 0) {\n                        atomicMin(minimum, numDeletedVertices);\n                    }\n                    __syncthreads();\n                } else {\n                    unsigned int goLeft = !((path >> k) & 1u);\n                    if(goLeft){\n                        \n                        if (threadIdx.x == 0){\n                            vertexDegrees_s[maxVertex] = -1;\n                        }\n                        ++numDeletedVertices;\n                        __syncthreads();\n\n                        for(unsigned int edge = graph.srcPtr[maxVertex] + threadIdx.x; edge < graph.srcPtr[maxVertex + 1]; edge+=blockDim.x) {\n                            unsigned int neighbor = graph.dst[edge];\n                            if(vertexDegrees_s[neighbor] != -1){\n                                #if USE_SHORT_DEGREE\n                                atomicSubShort(&vertexDegrees_s[neighbor], 1);\n                                #else\n                                atomicSub(&vertexDegrees_s[neighbor], 1);\n                                #endif\n                            }\n                        }\n\n                    } else {// Delete Neighbors of maxVertex\n                        numDeletedVertices += maxDegree;\n\n                        for(unsigned int edge = graph.srcPtr[maxVertex] + threadIdx.x; edge < graph.srcPtr[maxVertex + 1]; edge += blockDim.x) {\n                            unsigned int neighbor = graph.dst[edge];\n                            if(vertexDegrees_s[neighbor] != -1) {\n                                for(unsigned int neighborEdge = graph.srcPtr[neighbor]; neighborEdge < graph.srcPtr[neighbor + 1]; ++neighborEdge) {\n                                    unsigned int neighborOfNeighbor = graph.dst[neighborEdge];\n                                    if(vertexDegrees_s[neighborOfNeighbor] != -1) {\n                                        #if USE_SHORT_DEGREE\n                                        atomicSubShort(&vertexDegrees_s[neighborOfNeighbor], 1);\n                                        #else\n                                        atomicSub(&vertexDegrees_s[neighborOfNeighbor], 1);\n                                        #endif\n                                    }\n                                }\n                            }\n                        }\n                        __syncthreads();\n\n                        for(unsigned int edge = graph.srcPtr[maxVertex] + threadIdx.x; edge < graph.srcPtr[maxVertex + 1] ; edge += blockDim.x) {\n                            unsigned int neighbor = graph.dst[edge];\n                            vertexDegrees_s[neighbor] = -1;\n                        }\n                    }\n                }\n\n            }\n            __syncthreads();\n        }\n\n        // Each block its at it\'s required level which is at most Root depth\n        if(!vcFound && !minExceeded) {\n            unsigned int stackSize = (stacks.minimum + 1);\n            volatile INT_OR_SHORT * stackVertexDegrees = &stacks.stacks[(long long)blockIdx.x * (long long)stackSize * (long long)graph.vertexNum];\n            volatile unsigned int * stackNumDeletedVertices = &stacks.stacksNumDeletedVertices[blockIdx.x * stackSize];\n            volatile int * stackArrayIndex = &stacks.stackArrayIndex[blockIdx.x * stackSize];\n            volatile bool * stackCompNode = &stacks.stackCompNode[blockIdx.x * stackSize];\n\n            int arrayIndex = -1;\n            bool compNode = 0;\n            int stackTop = -1;\n\n            // go into while loop \n            bool popNextItr = false;\n            if (threadIdx.x == 0) {\n                s_current_depth = startingDepth;\n                s_last_branched_degree = 9999;\n            }\n            __syncthreads();\n            \n            do{\n                if(popNextItr) {\n                    startTime(POP_FROM_STACK,&blockCounters);\n                    popStack(graph.vertexNum, vertexDegrees_s, &numDeletedVertices, &arrayIndex, &compNode, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, stackCompNode, &stackTop, true);\n                    endTime(POP_FROM_STACK,&blockCounters);\n\n                    if (threadIdx.x == 0) {\n                        s_current_depth = startingDepth + stackTop + 1;\n                        s_last_branched_degree = 9999;\n                    }\n\n                    #if USE_COUNTERS\n                        if (threadIdx.x==0){\n                            atomicAdd(&NODES_PER_SM[sm_id],1);\n                        }\n                    #endif\n                }\n                __syncthreads();\n                \n                if(threadIdx.x == 0) {\n                    left = graph.vertexNum-1;\n                    right = 0;\n                }\n                __syncthreads();\n                \n                if (findBound) {\n                    startTime(FIND_BOUND,&blockCounters);\n                    findBoundaries(vertexDegrees_s, &left, &right, graph.vertexNum);\n                    endTime(FIND_BOUND,&blockCounters);\n                }\n\n                //reduction rule\n                __shared__ unsigned int minimum_s;\n                if(threadIdx.x == 0) {\n                    if (arrayIndex>-1) {\n                        #if USE_SHORT_DEGREE\n                        minimum_s = readIntFromShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum]);\n                        #else\n                        minimum_s = stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum];\n                        #endif\n                    } else {\n                        minimum_s = atomicOr(minimum,0);\n                    }\n                    \n                    if (right == 0) {\n                        left = 0;\n                        right = graph.vertexNum-1;\n                    }\n                }\n                __syncthreads();\n        \n                unsigned int iterationCounter = 0, numDeletedVerticesLeaf, numDeletedVerticesTriangle, numDeletedVerticesHighDegree;\n                do {\n                    startTime(LEAF_REDUCTION,&blockCounters);\n                    numDeletedVerticesLeaf = leafReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, left, right);\n                    endTime(LEAF_REDUCTION,&blockCounters);\n                    numDeletedVertices += numDeletedVerticesLeaf;\n                    if(iterationCounter==0 || numDeletedVerticesLeaf>0 || numDeletedVerticesHighDegree>0) {\n                        startTime(TRIANGLE_REDUCTION,&blockCounters);\n                        numDeletedVerticesTriangle = triangleReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, left, right);\n                        endTime(TRIANGLE_REDUCTION,&blockCounters);\n                        numDeletedVertices += numDeletedVerticesTriangle;\n                    } else {\n                        numDeletedVerticesTriangle = 0;\n                    }\n                    if(iterationCounter==0 || numDeletedVerticesLeaf>0 || numDeletedVerticesTriangle>0) {\n                        startTime(HIGH_DEGREE_REDUCTION,&blockCounters);\n                        numDeletedVerticesHighDegree = highDegreeReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2,numDeletedVertices,minimum_s, left, right);\n                        endTime(HIGH_DEGREE_REDUCTION,&blockCounters);\n                        numDeletedVertices += numDeletedVerticesHighDegree;\n                    } else {\n                        numDeletedVerticesHighDegree = 0;\n                    }\n                    ++iterationCounter;\n                } while(numDeletedVerticesTriangle>0 || numDeletedVerticesHighDegree>0);\n\n                startTime(NUM_EDGES,&blockCounters);\n                unsigned int numOfEdges = findNumOfEdges(graph.vertexNum, vertexDegrees_s, vertexDegrees_s2, left, right);\n                endTime(NUM_EDGES,&blockCounters);\n\n                bool numEdgesCondition = false;\n                if(numOfEdges>=square((int)minimum_s-(int)numDeletedVertices-1)+1) {\n                    numEdgesCondition = true;\n                }\n                __syncthreads();\n        \n                if(threadIdx.x == 0) {\n                    if (arrayIndex>-1) {\n                        #if USE_SHORT_DEGREE\n                        minimum_s = readIntFromShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum]);\n                        #else\n                        minimum_s = stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum];\n                        #endif\n                    } else {\n                        minimum_s = atomicOr(minimum,0);\n                    }\n                }\n                __syncthreads();\n\n                if(numDeletedVertices>=minimum_s || numEdgesCondition) {\n                    if (threadIdx.x == 0 && arrayIndex>-1 && compNode) {\n                        #if USE_SHORT_DEGREE\n                        int counter = readIntFromShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3]);\n                        saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3], counter-1);\n                        stackNumDeletedVertices[arrayIndex] += readIntFromShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum]);\n                        if (arrayIndex != stackTop) {\n                            unsigned int minim = *minimum;\n                            if (stackArrayIndex[arrayIndex]>-1) {\n                                minim = readIntFromShortArray(&stackVertexDegrees[(long long)stackArrayIndex[arrayIndex]*(long long)graph.vertexNum]);\n                            }\n                            if (minim>stackNumDeletedVertices[arrayIndex]+counter) {\n                                    saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum], minim-stackNumDeletedVertices[arrayIndex]-counter+1);\n                            } else {\n                                stackNumDeletedVertices[arrayIndex] += counter;\n                                saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum], 0);\n                            } \n                        } else {\n                            saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum], 0);\n                            saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3], 0);\n                        }\n                        #else\n                        int counter = stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3];\n                        stackVertexDegrees[arrayIndex*graph.vertexNum+3] = counter-1;\n                        stackNumDeletedVertices[arrayIndex] += stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum];\n                        if (arrayIndex != stackTop) {\n                            unsigned int minim = *minimum;\n                            if (stackArrayIndex[arrayIndex]>-1) {\n                                minim = stackVertexDegrees[(long long)stackArrayIndex[arrayIndex]*(long long)graph.vertexNum];\n                            }\n                            if (minim>stackNumDeletedVertices[arrayIndex]+counter) {\n                                    stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum] = minim-stackNumDeletedVertices[arrayIndex]-counter+1;\n                            } else {\n                                stackNumDeletedVertices[arrayIndex] += counter;\n                                stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum] = 0;\n                            } \n                        } else {\n                            stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum] = 0;\n                            stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3] = 0;\n                        }\n                        #endif\n                    }\n                    popNextItr = true;\n                    __syncthreads();\n                    if (threadIdx.x == 0) {\n                        run_cc_scan = (s_current_depth == 0) || \n                                      (s_last_branched_degree >= LAZY_TAU) || \n                                      (s_current_depth % LAZY_STRIDE == 0);\n                    }\n                    __syncthreads();\n\n                    unsigned int isComponents = 1;\n                    if (run_cc_scan) {\n                        #if USE_COUNTERS\n                        if (threadIdx.x == 0) {\n                            blockCounters.totalTime[TOTAL_BFS_SCANS]++;\n                        }\n                        #endif\n                        startTime(COMP_SEARCH,&blockCounters);\n                        isComponents = bfsCC_StackOnly(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, arrayIndex, compNode, minimum_s, &numDeletedVertices, &blockCounters, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, stackCompNode, &stackTop, stackSize, left, right);\n                        endTime(COMP_SEARCH,&blockCounters);\n                        #if USE_COUNTERS\n                        if (threadIdx.x == 0 && isComponents == 1) {\n                            blockCounters.totalTime[UNPRODUCTIVE_BFS_SCANS]++;\n                        }\n                        #endif\n                    } else {\n                        isComponents = 1;\n                    }\n                    __syncthreads();\n                    \n                    unsigned int maxVertex;\n                    INT_OR_SHORT maxDegree;\n                    if (isComponents>1) {\n                        popNextItr = true;\n                        __syncthreads();\n                        \n                    } else if (isComponents==0) {\n                        if(threadIdx.x == 0) {\n                            if (arrayIndex>-1) {\n                                #if USE_SHORT_DEGREE\n                                if (numDeletedVertices<readIntFromShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum])) {\n                                    saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum], numDeletedVertices);\n                                }\n                                if (compNode) {\n                                    int counter = readIntFromShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3]);\n                                    saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3], counter-1);\n                                    stackNumDeletedVertices[arrayIndex] += readIntFromShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum]);\n                                    if (arrayIndex != stackTop) {\n                                        unsigned int minim = *minimum;\n                                        if (stackArrayIndex[arrayIndex]>-1) {\n                                            minim = readIntFromShortArray(&stackVertexDegrees[(long long)stackArrayIndex[arrayIndex]*(long long)graph.vertexNum]);\n                                        }\n                                        if (minim>stackNumDeletedVertices[arrayIndex]+counter) {\n                                                saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum], minim-stackNumDeletedVertices[arrayIndex]-counter+1);\n                                        } else {\n                                            stackNumDeletedVertices[arrayIndex] += counter;\n                                            saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum], 0);\n                                        }\n                                    } else {\n                                        saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum], 0);\n                                        saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3], 0);\n                                    }\n                                }\n                                #else\n                                if (numDeletedVertices<stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum]) {\n                                    stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum] = numDeletedVertices;\n                                }\n                                if (compNode) {\n                                    int counter = stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3];\n                                    stackVertexDegrees[arrayIndex*graph.vertexNum+3] = counter-1;\n                                    stackNumDeletedVertices[arrayIndex] += stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum];\n                                    if (arrayIndex != stackTop) {\n                                        unsigned int minim = *minimum;\n                                        if (stackArrayIndex[arrayIndex]>-1) {\n                                            minim = stackVertexDegrees[(long long)stackArrayIndex[arrayIndex]*(long long)graph.vertexNum];\n                                        }\n                                        if (minim>stackNumDeletedVertices[arrayIndex]+counter) {\n                                                stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum] = minim-stackNumDeletedVertices[arrayIndex]-counter+1;\n                                        } else {\n                                            stackNumDeletedVertices[arrayIndex] += counter;\n                                            stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum] = 0;\n                                        }\n                                    } else {\n                                        stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum] = 0;\n                                        stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3] = 0;\n                                    }\n                                }\n                                #endif\n                            } else {\n                                atomicMin(minimum, numDeletedVertices);\n                            }\n                        }\n                        popNextItr = true;\n                        __syncthreads();\n                    } else {\n                    \n                        // Find max degree\n                        startTime(MAX_DEGREE,&blockCounters);\n                        findMaxDegree(graph.vertexNum, &maxVertex, &maxDegree, vertexDegrees_s, vertexDegrees_s2, left, right);\n                        endTime(MAX_DEGREE,&blockCounters);\n\n                        startTime(PREPARE_RIGHT_CHILD,&blockCounters);\n                        deleteNeighborsOfMaxDegreeVertex(graph,vertexDegrees_s, &numDeletedVertices, vertexDegrees_s2, &numDeletedVertices2, maxDegree, maxVertex);\n                        endTime(PREPARE_RIGHT_CHILD,&blockCounters);\n\n                        startTime(PUSH_TO_STACK,&blockCounters);\n                        pushStack(graph.vertexNum, vertexDegrees_s2, &numDeletedVertices2, arrayIndex, compNode, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, stackCompNode, &stackTop, true);\n                        endTime(PUSH_TO_STACK,&blockCounters);\n                        maxDepth(stackTop, &blockCounters);\n                        __syncthreads();\n\n                        startTime(PREPARE_LEFT_CHILD,&blockCounters);\n                        deleteMaxDegreeVertex(graph, vertexDegrees_s, &numDeletedVertices, maxVertex);\n                        endTime(PREPARE_LEFT_CHILD,&blockCounters);\n\n                        #if USE_COUNTERS\n                            if (threadIdx.x==0){\n                                atomicAdd(&NODES_PER_SM[sm_id],1);\n                            }\n                        #endif\n\n                        compNode = 0;\n                        popNextItr = false;\n                        if (threadIdx.x == 0) {\n                            s_current_depth++;\n                            s_last_branched_degree = maxDegree;\n                        }\n                        __syncthreads();\n                    }\n                }\n                __syncthreads();\n\n            } while(stackTop != -1);\n        }\n    }while(true);\n\n    #if USE_COUNTERS\n    if(threadIdx.x == 0) {\n        counters[blockIdx.x] = blockCounters;\n    }\n    #endif\n}\n\n#undef INT_OR_SHORT\n#undef FUNCTION_NAME(base)\n#undef SHARED_MEM_TYPE(base)')

# Inject optimized Variant D files (Variant C + Addition 2: Analytic Tree & Cycle Fast-Path)
print("Deploying Variant D (Addition 1 + Addition 2: Lazy + Analytic Fast-Path) optimizations...")
with open("solver_variant_d_full/src/Counters.cuh", "w") as f:
    f.write('#ifndef COUNTERS_H\n#define COUNTERS_H\n\n#include <string.h>\n#include <sys/stat.h>\n#include "config.h"\n\nenum CounterName { ENQUEUE=0, DEQUEUE, FIND_BOUND, LEAF_REDUCTION, HIGH_DEGREE_REDUCTION, TRIANGLE_REDUCTION, COMP_SEARCH, MAX_DEGREE, NUM_EDGES, PREPARE_LEFT_CHILD, PREPARE_RIGHT_CHILD, PUSH_TO_STACK, POP_FROM_STACK, TERMINATE, MAX_DEPTH, TOTAL_BFS_SCANS, UNPRODUCTIVE_BFS_SCANS, CYCLE_FAST_PATH_HITS, TREE_FAST_PATH_HITS, NUM_COUNTERS };\n\nstruct Counters {\n    unsigned long long tmp[NUM_COUNTERS];\n    unsigned long long totalTime[NUM_COUNTERS];\n};\n\nstatic __device__ void initializeCounters(Counters* counters) {\n    #if USE_COUNTERS\n    if(threadIdx.x == 0) {\n        for(unsigned int i = 0; i < NUM_COUNTERS; ++i) {\n            counters->totalTime[i] = 0;\n        }\n    }\n    #endif\n}\n\nstatic __device__ void maxDepth(int stackTop, Counters* counters) {\n    #if USE_COUNTERS\n        if(threadIdx.x == 0) {\n            if(counters->totalTime[MAX_DEPTH] < stackTop){\n                counters->totalTime[MAX_DEPTH] = stackTop;\n            }\n        }\n    #endif\n}\n\nstatic __device__ void startTime(CounterName counterName, Counters* counters) {\n#if USE_COUNTERS\n    if(threadIdx.x == 0) {\n        counters->tmp[counterName] = clock64();\n    }\n#endif\n}\n\nstatic __device__ void endTime(CounterName counterName, Counters* counters) {\n#if USE_COUNTERS\n    if(threadIdx.x == 0) {\n        counters->totalTime[counterName] += clock64() - counters->tmp[counterName];\n    }\n#endif\n}\n\nvoid printCountersInFile(Config config, Counters * counters_d, unsigned int numBlocks, double reduceAndInduceTime, double kernelTime){\n\n    Counters * counters;\n    counters = (Counters *)malloc(sizeof(Counters)*numBlocks);\n    cudaMemcpy(counters, counters_d, sizeof(Counters)*numBlocks, cudaMemcpyDeviceToHost);\n\n    char filename[100];\n    strcpy(filename,config.graphFileName);\n\n    char * token = strtok(filename, "/");\n    // loop through the string to extract all other tokens\n    char * outputfile;\n    while( token != NULL ) {\n        outputfile = token;\n        token = strtok(NULL, "/");\n    }\n\n    char outputFilename[500];\n    strcpy(outputFilename,"Counters/Counters_");\n    strcat(outputFilename,outputfile);\n\n    mkdir("Counters", 0777);\n    FILE *output_file = fopen(outputFilename, "w"); \n\n    fprintf(output_file,"BLOCK_NO,Add to worklist,Remove from worklist,Find boundaries,Degree-one rule,High-degree rule,Degree-two-triangle rule,Components search,Find max degree vertex,Stopping condition,Remove max-degree vertex,Remove neighbors of max-degree vertex,Push to stack,Pop from stack,Terminate,MAX_DEPTH,Total BFS Scans,Unproductive BFS Scans,Cycle Fast-Path Hits,Tree Fast-Path Hits\\n");\n \n    for(unsigned int i = 0;i<numBlocks;++i){\n        fprintf(output_file,"%u",i);\n        for(unsigned int j = 0;j<NUM_COUNTERS;++j){\n            fprintf(output_file,",%llu",counters[i].totalTime[j]);\n        }\n        fprintf(output_file,"\\n");\n    }\n\n    fprintf(output_file, "\\n");\n    fprintf(output_file, "Reduce and Induce time: %fms\\n", reduceAndInduceTime);\n    fprintf(output_file, "Kernel time: %fms\\n", kernelTime);\n\n    free(counters);\n    fclose(output_file);\n}\n\n__device__ int get_smid(void) {\n    uint ret;\n    asm("mov.u32 %0, %smid;" : "=r"(ret) );\n    return ret;\n}\n\nvoid printNodesPerSM(Config config, unsigned long long * NODES_PER_SM,  int numOfSMs){\n\n    unsigned long long * NODES_PER_SM_host;\n    NODES_PER_SM_host = (unsigned long long *)malloc(sizeof(unsigned long long)*numOfSMs);\n    cudaMemcpy(NODES_PER_SM_host, NODES_PER_SM, sizeof(unsigned long long)*numOfSMs, cudaMemcpyDeviceToHost);\n\n    char filename[100];\n    strcpy(filename,config.graphFileName);\n\n    char * token = strtok(filename, "/");\n    // loop through the string to extract all other tokens\n    char * outputfile;\n    while( token != NULL ) {\n        outputfile = token;\n        token = strtok(NULL, "/");\n    }\n\n    char outputFilename[500];\n    strcpy(outputFilename,"NODES_PER_SM/NODES_PER_SM_");\n    strcat(outputFilename,outputfile);\n\n    mkdir("NODES_PER_SM", 0777);\n    FILE *output_file = fopen(outputFilename, "w");\n\n\tfor(unsigned int i = 0;i<numOfSMs;++i){\n\t\tchar str[100] = "SM_";\n\t\tfprintf(output_file,"SM_%u",i+1);\n\t\tif (i < numOfSMs - 1){\n\t\t\tfprintf(output_file,",");\n\t\t}\n\t}\n\tfprintf(output_file,"\\n");\n \n    for(unsigned int i = 0;i<numOfSMs;++i){\n\t\tfprintf(output_file,"%llu",NODES_PER_SM_host[i]);\n\t\tif (i < numOfSMs - 1){\n\t\t\tfprintf(output_file,",");\n\t\t}\n    }\n\n    free(NODES_PER_SM_host);\n    fclose(output_file);\n}\n\n\n#endif\n')

with open("solver_variant_d_full/src/GlobalWorkList.cuh", "w") as f:
    f.write('#include <stdio.h>\n#include <stdint.h>\n#include <assert.h>\n\n#include "config.h"\n#include "stack.cuh"\n#include "Counters.cuh"\n#include "BWDWorkList.cuh"\n#include "helperFunctions.cuh"\n\n#ifndef LAZY_TAU\n#define LAZY_TAU 4        // Hub-removal degree threshold (tau)\n#endif\n\n#ifndef LAZY_STRIDE\n#define LAZY_STRIDE 2     // Search-tree depth stride (S)\n#endif\n\n#if USE_SHORT_DEGREE\n#define INT_OR_SHORT short\n#define FUNCTION_NAME(base) base##_short\n#define SHARED_MEM_TYPE(base) base##_short\n#else\n#define INT_OR_SHORT int\n#define FUNCTION_NAME(base) base##_int\n#define SHARED_MEM_TYPE(base) base##_int\n#endif\n\n#if USE_GLOBAL_MEMORY\n__global__ void FUNCTION_NAME(GlobalWorkList_global_kernel)(Stacks<INT_OR_SHORT> stacks, unsigned int * minimum, WorkList<INT_OR_SHORT> workList, CSRGraph graph, Counters* counters, int* first_to_dequeue_global, INT_OR_SHORT* global_memory, unsigned long long* NODES_PER_SM, CompArray compArray, unsigned int* compArraySize, unsigned int findBound, unsigned int* numBranches, unsigned int* numBranchesFreq) {\n#else\n__global__ void FUNCTION_NAME(GlobalWorkList_shared_kernel)(Stacks<INT_OR_SHORT> stacks, unsigned int * minimum, WorkList<INT_OR_SHORT> workList, CSRGraph graph, Counters* counters, int* first_to_dequeue_global, unsigned long long* NODES_PER_SM, CompArray compArray, unsigned int* compArraySize, unsigned int findBound, unsigned int* numBranches, unsigned int* numBranchesFreq) {\n#endif\n\n    // For Evaluation Purposes\n    //__shared__ Counters blockCounters;\n \n    Counters blockCounters;\n    initializeCounters(&blockCounters);\n    \n    // For Evaluation Purposes\n    #if RUN_KILL_TIMEOUT\n    __shared__ long long startingTime;\n    __shared__ bool timeOut;\n    if (threadIdx.x==0) {\n        startingTime = clock64();\n        timeOut = false;\n    }\n    #endif\n\n    #if USE_COUNTERS\n        __shared__ unsigned int sm_id;\n        if (threadIdx.x==0){\n            sm_id=get_smid();\n        }\n    #endif\n    int stackTop = -1;\n    unsigned int stackSize = (stacks.minimum + 1);\n    volatile INT_OR_SHORT * stackVertexDegrees = &stacks.stacks[(long long)blockIdx.x * (long long)stackSize * (long long)graph.vertexNum];\n    volatile unsigned int * stackNumDeletedVertices = &stacks.stacksNumDeletedVertices[blockIdx.x * stackSize];\n    volatile int * stackArrayIndex = &stacks.stackArrayIndex[blockIdx.x * stackSize];\n\n    __shared__ int s_current_depth;\n    __shared__ int s_last_branched_degree;\n    __shared__ bool run_cc_scan;\n\n    // Define the vertexDegree_s\n    unsigned int numDeletedVertices;\n    unsigned int numDeletedVertices2;\n    \n    int arrayIndex = -1;\n    \n    #if USE_GLOBAL_MEMORY\n    INT_OR_SHORT * vertexDegrees_s = &global_memory[graph.vertexNum*(2*blockIdx.x)];\n    INT_OR_SHORT * vertexDegrees_s2 = &global_memory[graph.vertexNum*(2*blockIdx.x + 1)];\n    #else\n    extern __shared__ INT_OR_SHORT SHARED_MEM_TYPE(shared_mem)[];\n    INT_OR_SHORT * vertexDegrees_s = SHARED_MEM_TYPE(shared_mem);\n    INT_OR_SHORT * vertexDegrees_s2 = &SHARED_MEM_TYPE(shared_mem)[graph.vertexNum];\n    #endif\n    \n    bool dequeueOrPopNextItr = true; \n    __syncthreads();\n\n    __shared__ bool first_to_dequeue;\n    if (threadIdx.x==0) {\n        if(atomicCAS(first_to_dequeue_global,0,1) == 0) { \n            first_to_dequeue = true;\n        } else {\n            first_to_dequeue = false;\n        }\n    }\n    __syncthreads();\n    if (first_to_dequeue) {\n        for(unsigned int vertex = threadIdx.x; vertex < graph.vertexNum; vertex += blockDim.x) {\n            vertexDegrees_s[vertex] = graph.degree[vertex];\n        }\n        numDeletedVertices = 0;\n        dequeueOrPopNextItr = false;\n        if (threadIdx.x == 0) {\n            s_current_depth = 0;\n            s_last_branched_degree = 9999;\n        }\n        #if USE_COUNTERS\n            if (threadIdx.x==0){\n                atomicAdd(&NODES_PER_SM[sm_id],1);\n            }\n        #endif\n    }\n    __syncthreads();\n\n    while(true) {\n        if(dequeueOrPopNextItr) {\n            if(stackTop != -1) { // Local stack is not empty, pop from the local stack\n                startTime(POP_FROM_STACK,&blockCounters);\n                popStack(graph.vertexNum, vertexDegrees_s, &numDeletedVertices, &arrayIndex, NULL, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, NULL, &stackTop, false);\n                endTime(POP_FROM_STACK,&blockCounters);\n\n                if (threadIdx.x == 0) {\n                    s_current_depth = stackTop + 1;\n                    s_last_branched_degree = 9999;\n                }\n\n                #if USE_COUNTERS\n                    if (threadIdx.x==0){\n                        atomicAdd(&NODES_PER_SM[sm_id],1);\n                    }\n                #endif\n            } else { // Local stack is empty, read from the global workList\n                startTime(TERMINATE,&blockCounters);\n                startTime(DEQUEUE,&blockCounters);\n                if(!dequeue(vertexDegrees_s, workList, graph.vertexNum, &numDeletedVertices, &arrayIndex)) {   \n                    endTime(TERMINATE,&blockCounters);\n                    break;\n                }\n                endTime(DEQUEUE,&blockCounters);\n\n                if (threadIdx.x == 0) {\n                    s_current_depth = 0;\n                    s_last_branched_degree = 9999;\n                }\n                \n                // For Debugging Purposes\n                /*__shared__ bool conditionn;\n                if (threadIdx.x==0) {\n                    conditionn = false;\n                    for (int i = 0; i<graph.vertexNum; ++i) {\n                        if (vertexDegrees_s[i]>0) {\n                            unsigned int counter = 0;\n                            for(unsigned int edge = graph.srcPtr[i]; edge<graph.srcPtr[i+1]; ++edge) {\n                                unsigned int neigh = graph.dst[edge];\n                                if(vertexDegrees_s[neigh]>0) {\n                                    ++counter;\n                                }\n                            }\n                            if (counter != vertexDegrees_s[i]) {\n                                conditionn = true;\n                            }\n                        }\n                    }\n                    if (conditionn) {\n                        printf("Head = %u, Tail = %u, woklist Baaaaaaaaaaaaaad count = %d, numWait = %d, numEnq = %d\\n",\n                           (unsigned int)(*workList.head_tail & 0xFFFFFFFF),\n                           (unsigned int)((*workList.head_tail >> 32) & 0xFFFFFFFF),\n                           *workList.count,\n                           workList.counter->numWaiting,\n                           workList.counter->numEnqueued\n                        );\n                    } else {\n                        printf("Head = %u, Tail = %u, woklist count = %d, numWait = %d, numEnq = %d\\n",\n                           (unsigned int)(*workList.head_tail & 0xFFFFFFFF),\n                           (unsigned int)((*workList.head_tail >> 32) & 0xFFFFFFFF),\n                           *workList.count,\n                           workList.counter->numWaiting,\n                           workList.counter->numEnqueued\n                        );\n                    }\n                }\n                __syncthreads();*/\n\n                #if USE_COUNTERS\n                    if (threadIdx.x==0){\n                        atomicAdd(&NODES_PER_SM[sm_id],1);\n                    }\n                #endif\n            }\n        }\n        __syncthreads();\n        \n        __shared__ unsigned int left;\n        __shared__ unsigned int right;\n        if(threadIdx.x == 0){\n            left = graph.vertexNum-1;\n            right = 0;\n        }\n        __syncthreads();\n        \n        if (findBound) {\n            startTime(FIND_BOUND,&blockCounters);\n            findBoundaries(vertexDegrees_s, &left, &right, graph.vertexNum);\n            endTime(FIND_BOUND,&blockCounters);\n        }\n\n        __shared__ unsigned int minimum_s;\n        if(threadIdx.x == 0) {\n            if (arrayIndex>-1) {\n                minimum_s = atomicOr((int*)&compArray.entries[arrayIndex].minimum,0);\n            } else {\n                minimum_s = atomicOr(minimum,0);\n            }\n            \n            if (right == 0) {\n                left = 0;\n                right = graph.vertexNum-1;\n            }\n            \n            // For Evaluation Purposes\n            #if RUN_KILL_TIMEOUT\n            if (clock64() - startingTime > 25000000000000) {\n                timeOut = true;\n            }\n            #endif\n        }\n        __syncthreads();\n\n        // For Evaluation Purposes\n        #if RUN_KILL_TIMEOUT\n        if (timeOut) {\n            stackTop = -1;\n            dequeueOrPopNextItr = true;\n            continue;\n        }\n        #endif\n\n        unsigned int iterationCounter = 0, numDeletedVerticesLeaf, numDeletedVerticesTriangle, numDeletedVerticesHighDegree;\n        do {\n            startTime(LEAF_REDUCTION,&blockCounters);\n            numDeletedVerticesLeaf = leafReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, left, right);\n            endTime(LEAF_REDUCTION,&blockCounters);\n            numDeletedVertices += numDeletedVerticesLeaf;\n            if(iterationCounter==0 || numDeletedVerticesLeaf>0 || numDeletedVerticesHighDegree>0) {\n                startTime(TRIANGLE_REDUCTION,&blockCounters);\n                numDeletedVerticesTriangle = triangleReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, left, right);\n                endTime(TRIANGLE_REDUCTION,&blockCounters);\n                numDeletedVertices += numDeletedVerticesTriangle;\n            } else {\n                numDeletedVerticesTriangle = 0;\n            }\n            if(iterationCounter==0 || numDeletedVerticesLeaf>0 || numDeletedVerticesTriangle>0) {\n                startTime(HIGH_DEGREE_REDUCTION,&blockCounters);\n                numDeletedVerticesHighDegree = highDegreeReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2,numDeletedVertices,minimum_s, left, right);\n                endTime(HIGH_DEGREE_REDUCTION,&blockCounters);\n                numDeletedVertices += numDeletedVerticesHighDegree;\n            } else {\n                numDeletedVerticesHighDegree = 0;\n            }\n            ++iterationCounter;\n        } while(numDeletedVerticesTriangle>0 || numDeletedVerticesHighDegree>0);\n        \n        startTime(NUM_EDGES,&blockCounters);\n        unsigned int numOfEdges = findNumOfEdges(graph.vertexNum, vertexDegrees_s, vertexDegrees_s2, left, right);\n        endTime(NUM_EDGES,&blockCounters);\n\n        bool numEdgesCondition = false;\n        if(numOfEdges>=square((int)minimum_s-(int)numDeletedVertices-1)+1) {\n            numEdgesCondition = true;\n        }\n        __syncthreads();\n\n        if(threadIdx.x == 0) {\n            if (arrayIndex>-1) {\n                minimum_s = atomicOr((int*)&compArray.entries[arrayIndex].minimum,0);\n            } else {\n                minimum_s = atomicOr(minimum,0);\n            }\n            // For Debugging Purposes\n            //printf("I am Alive blockIdx.x = %d and stackTop = %d and minim = %u\\n", blockIdx.x, stackTop, minimum_s);\n        }\n        __syncthreads();\n\n        if(numDeletedVertices>=minimum_s || numEdgesCondition) { // Reached the bottom of the tree, no minimum vertex cover found\n            if(threadIdx.x==0 && arrayIndex>-1) {\n                int counter = atomicSub((int*)&compArray.entries[arrayIndex].counter, 1);\n                if (counter==1) {\n                    int parentIndex = compArray.entries[arrayIndex].index;\n                    atomicAdd((int*)&compArray.entries[parentIndex].minimum, compArray.entries[arrayIndex].minimum);\n                    int parentCounter = atomicSub((int*)&compArray.entries[parentIndex].counter, 1);\n                    if (parentCounter==1) {\n                        arrayIndex = compArray.entries[parentIndex].index;\n                        numDeletedVertices = atomicOr((int*)&compArray.entries[parentIndex].minimum,0);\n                        while (true) {\n                            if (arrayIndex>-1) {\n                                atomicMin((int*)&compArray.entries[arrayIndex].minimum, numDeletedVertices);\n                                int counter = atomicSub((int*)&compArray.entries[arrayIndex].counter, 1);\n                                if (counter-1>0) {\n                                    break;\n                                }\n                                int parentIndex = compArray.entries[arrayIndex].index;\n                                atomicAdd((int*)&compArray.entries[parentIndex].minimum, compArray.entries[arrayIndex].minimum);\n                                int parentCounter = atomicSub((int*)&compArray.entries[parentIndex].counter, 1);\n                                if (parentCounter-1>0) {\n                                    break;\n                                }\n                                arrayIndex = compArray.entries[parentIndex].index;\n                                numDeletedVertices = atomicOr((int*)&compArray.entries[parentIndex].minimum,0);\n\n                            } else {\n                                atomicMin(minimum, numDeletedVertices);\n                                break;\n                            }\n                        }\n                    }\n                }\n            }\n            dequeueOrPopNextItr = true;\n            __syncthreads();\n\n        } else {\n\n            if (threadIdx.x == 0) {\n                run_cc_scan = (s_current_depth == 0) || \n                              (s_last_branched_degree >= LAZY_TAU) || \n                              (s_current_depth % LAZY_STRIDE == 0);\n            }\n            __syncthreads();\n\n            int isComponents = 1;\n            if (run_cc_scan) {\n                #if USE_COUNTERS\n                if (threadIdx.x == 0) {\n                    blockCounters.totalTime[TOTAL_BFS_SCANS]++;\n                }\n                #endif\n                startTime(COMP_SEARCH,&blockCounters);\n                isComponents = bfsCC_Hybrid(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, arrayIndex, minimum_s, &numDeletedVertices, compArray, workList, &blockCounters, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, &stackTop, stackSize, *compArraySize, left, right, minimum);\n                endTime(COMP_SEARCH,&blockCounters);\n                #if USE_COUNTERS\n                if (threadIdx.x == 0 && isComponents == 1) {\n                    blockCounters.totalTime[UNPRODUCTIVE_BFS_SCANS]++;\n                }\n                #endif\n            } else {\n                isComponents = 1;\n            }\n            __syncthreads();\n\n            unsigned int maxVertex;\n            INT_OR_SHORT maxDegree;\n            if (isComponents>1) {\n                #if COUNT_BRANCHING_ON_COMPONENTS\n                if (threadIdx.x==0) {\n                   atomicAdd(numBranches, 1);\n                   atomicAdd(&numBranchesFreq[isComponents], 1);\n                }\n                __syncthreads();\n                #endif\n                dequeueOrPopNextItr = true;\n                __syncthreads();\n\n            } else if (isComponents==0) { // Reached the bottom of the tree, minimum vertex cover possibly found\n                if(threadIdx.x==0) {\n                    while (true) {\n                        if (arrayIndex>-1) {\n                            atomicMin((int*)&compArray.entries[arrayIndex].minimum, numDeletedVertices);\n                            int counter = atomicSub((int*)&compArray.entries[arrayIndex].counter, 1);\n                            if (counter-1>0) {\n                                break;\n                            }\n                            int parentIndex = compArray.entries[arrayIndex].index;\n                            atomicAdd((int*)&compArray.entries[parentIndex].minimum, compArray.entries[arrayIndex].minimum);\n                            int parentCounter = atomicSub((int*)&compArray.entries[parentIndex].counter, 1);\n                            if (parentCounter-1>0) {\n                                break;\n                            }\n                            arrayIndex = compArray.entries[parentIndex].index;\n                            numDeletedVertices = atomicOr((int*)&compArray.entries[parentIndex].minimum,0);\n                        } else {\n                            atomicMin(minimum, numDeletedVertices);\n                            break;\n                        }\n                    }\n                }\n                dequeueOrPopNextItr = true;\n                __syncthreads();\n\n            } else {\n                // Vertex cover not found, need to branch\n                if(threadIdx.x == 0) {\n                    if (arrayIndex>-1) {\n                        atomicAdd((int*)&compArray.entries[arrayIndex].counter, 1);\n                    }\n                }\n                __syncthreads();\n                \n                startTime(MAX_DEGREE,&blockCounters);\n                findMaxDegree(graph.vertexNum, &maxVertex, &maxDegree, vertexDegrees_s, vertexDegrees_s2, left, right);\n                endTime(MAX_DEGREE,&blockCounters);\n\n                startTime(PREPARE_RIGHT_CHILD,&blockCounters);\n                deleteNeighborsOfMaxDegreeVertex(graph,vertexDegrees_s, &numDeletedVertices, vertexDegrees_s2, &numDeletedVertices2, maxDegree, maxVertex);\n                endTime(PREPARE_RIGHT_CHILD,&blockCounters);\n\n                bool enqueueSuccess;\n                if(checkThreshold(workList)) {\n                    startTime(ENQUEUE,&blockCounters);\n                    enqueueSuccess = enqueue(vertexDegrees_s2, workList, graph.vertexNum, &numDeletedVertices2, arrayIndex);\n                } else  {\n                    enqueueSuccess = false;\n                }\n                __syncthreads();\n\n                if(!enqueueSuccess) {\n                    startTime(PUSH_TO_STACK,&blockCounters);\n                    pushStack(graph.vertexNum, vertexDegrees_s2, &numDeletedVertices2, arrayIndex, NULL, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, NULL, &stackTop, false);\n                    endTime(PUSH_TO_STACK,&blockCounters);\n                    maxDepth(stackTop, &blockCounters);\n                } else {\n                    endTime(ENQUEUE,&blockCounters);\n                }\n                __syncthreads();\n\n                startTime(PREPARE_LEFT_CHILD,&blockCounters);\n                // Prepare the child that removes the neighbors of the max vertex to be processed on the next iteration\n                deleteMaxDegreeVertex(graph, vertexDegrees_s, &numDeletedVertices, maxVertex);\n                endTime(PREPARE_LEFT_CHILD,&blockCounters);\n\n                #if USE_COUNTERS\n                    if (threadIdx.x==0) {\n                        atomicAdd(&NODES_PER_SM[sm_id],1);\n                    }\n                #endif\n\n                dequeueOrPopNextItr = false;\n                if (threadIdx.x == 0) {\n                    s_current_depth++;\n                    s_last_branched_degree = maxDegree;\n                }\n                __syncthreads();\n            }\n        }\n        __syncthreads();\n    }\n\n    // For Debugging Purposes\n    /*if (blockIdx.x==0) {\n        if (threadIdx.x==0) {\n            int globIndex = (*compArray.globalArrayIndex<*compArraySize)?*compArray.globalArrayIndex:*compArraySize;\n            for(unsigned int vertex = 0; vertex<globIndex; vertex++) {\n                printf("index = %d\\n",vertex);\n                printf("counter = %d\\n",compArray.entries[vertex].counter);\n                printf("minimum = %d\\n",compArray.entries[vertex].minimum);\n                printf("index = %d\\n\\n",compArray.entries[vertex].index);\n            }\n        }\n    }*/\n\n    #if USE_COUNTERS\n    if(threadIdx.x == 0) {\n        counters[blockIdx.x] = blockCounters;\n    }\n    #endif\n}\n\n#undef INT_OR_SHORT\n#undef FUNCTION_NAME(base)\n#undef SHARED_MEM_TYPE(base)')

with open("solver_variant_d_full/src/LocalStacks.cuh", "w") as f:
    f.write('#include <stdio.h>\n#include <stdint.h>\n#include <assert.h>\n\n#include "config.h"\n#include "stack.cuh"\n#include "Counters.cuh"\n#include "helperFunctions.cuh"\n\n#ifndef LAZY_TAU\n#define LAZY_TAU 4        // Hub-removal degree threshold (tau)\n#endif\n\n#ifndef LAZY_STRIDE\n#define LAZY_STRIDE 2     // Search-tree depth stride (S)\n#endif\n\n#if USE_SHORT_DEGREE\n#define INT_OR_SHORT short\n#define FUNCTION_NAME(base) base##_short\n#define SHARED_MEM_TYPE(base) base##_short\n#else\n#define INT_OR_SHORT int\n#define FUNCTION_NAME(base) base##_int\n#define SHARED_MEM_TYPE(base) base##_int\n#endif\n\n#if USE_GLOBAL_MEMORY\n__global__ void FUNCTION_NAME(LocalStacks_global_kernel)(Stacks<INT_OR_SHORT> stacks, CSRGraph graph, unsigned int * minimum, INT_OR_SHORT* global_memory, Counters* counters, unsigned int * pathCounter, unsigned long long* NODES_PER_SM, int startingDepth, unsigned int findBound) {\n#else\n__global__ void FUNCTION_NAME(LocalStacks_shared_kernel)(Stacks<INT_OR_SHORT> stacks, CSRGraph graph, unsigned int * minimum, Counters* counters, unsigned int * pathCounter, unsigned long long* NODES_PER_SM, int startingDepth, unsigned int findBound) {\n#endif\n\n    Counters blockCounters;\n    initializeCounters(&blockCounters);\n    \n    __shared__ unsigned int left;\n    __shared__ unsigned int right;\n    __shared__ int s_current_depth;\n    __shared__ int s_last_branched_degree;\n    __shared__ bool run_cc_scan;\n\n    #if USE_COUNTERS\n        __shared__ unsigned int sm_id;\n        if (threadIdx.x==0){\n            sm_id=get_smid();\n        }\n    #endif\n\n    do{\n        __shared__ unsigned int path;\n        if(threadIdx.x==0){\n            path = atomicAdd(pathCounter,1);\n        }\n        __syncthreads();\n\n        if(path>=(1<<startingDepth)){\n            break;\n        }\n\n        // Initialize the vertexDegrees_s\n        \n        unsigned int numDeletedVertices = 0;\n        unsigned int numDeletedVertices2;\n\n        #if USE_GLOBAL_MEMORY\n        INT_OR_SHORT * vertexDegrees_s = &global_memory[graph.vertexNum*(2*blockIdx.x)];\n        INT_OR_SHORT * vertexDegrees_s2 = &global_memory[graph.vertexNum*(2*blockIdx.x + 1)];\n        #else\n        extern __shared__ INT_OR_SHORT SHARED_MEM_TYPE(shared_mem)[];\n        INT_OR_SHORT * vertexDegrees_s = SHARED_MEM_TYPE(shared_mem);\n        INT_OR_SHORT * vertexDegrees_s2 = &SHARED_MEM_TYPE(shared_mem)[graph.vertexNum];\n        #endif\n\n        for(unsigned int i = threadIdx.x; i < graph.vertexNum; i += blockDim.x) {\n            vertexDegrees_s[i] = graph.degree[i];\n        }\n        __syncthreads();\n\n        // Find the block\'s sub-tree\n        bool vcFound = false;\n        bool minExceeded = false;\n        \n        for(unsigned int k = 0; k < startingDepth && !vcFound && !minExceeded; ++k) {\n            #if USE_COUNTERS\n                if (threadIdx.x==0){\n                    atomicAdd(&NODES_PER_SM[sm_id],1);\n                }\n            #endif\n            \n            if(threadIdx.x == 0){\n                left = graph.vertexNum-1;\n                right = 0;\n            }\n            __syncthreads();\n            if (k > 0) {\n                if (findBound) {\n                    startTime(FIND_BOUND,&blockCounters);\n                    findBoundaries(vertexDegrees_s, &left, &right, graph.vertexNum);\n                    endTime(FIND_BOUND,&blockCounters);\n                }\n            } else {\n                if(threadIdx.x == 0) {\n                    left = 0;\n                    right = graph.vertexNum -1;\n                }\n            }\n            __syncthreads();\n\n            // reduction rule\n            __shared__ unsigned int minimum_s;\n\n            if(threadIdx.x == 0) {\n                minimum_s = atomicOr(minimum,0);\n                if (right == 0) {\n                    left = 0;\n                    right = graph.vertexNum-1;\n                }\n            }\n            __syncthreads();\n\n            unsigned int iterationCounter = 0, numDeletedVerticesLeaf, numDeletedVerticesTriangle, numDeletedVerticesHighDegree;\n            do {\n                startTime(LEAF_REDUCTION,&blockCounters);\n                numDeletedVerticesLeaf = leafReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, left, right);\n                endTime(LEAF_REDUCTION,&blockCounters);\n                numDeletedVertices += numDeletedVerticesLeaf;\n                if(iterationCounter==0 || numDeletedVerticesLeaf>0 || numDeletedVerticesHighDegree>0) {\n                    startTime(TRIANGLE_REDUCTION,&blockCounters);\n                    numDeletedVerticesTriangle = triangleReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, left, right);\n                    endTime(TRIANGLE_REDUCTION,&blockCounters);\n                    numDeletedVertices += numDeletedVerticesTriangle;\n                } else {\n                    numDeletedVerticesTriangle = 0;\n                }\n                if(iterationCounter==0 || numDeletedVerticesLeaf>0 || numDeletedVerticesTriangle>0) {\n                    startTime(HIGH_DEGREE_REDUCTION,&blockCounters);\n                    numDeletedVerticesHighDegree = highDegreeReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2,numDeletedVertices,minimum_s, left, right);\n                    endTime(HIGH_DEGREE_REDUCTION,&blockCounters);\n                    numDeletedVertices += numDeletedVerticesHighDegree;\n                } else {\n                    numDeletedVerticesHighDegree = 0;\n                }\n                ++iterationCounter;\n            } while(numDeletedVerticesTriangle>0 || numDeletedVerticesHighDegree>0);\n\n            startTime(NUM_EDGES,&blockCounters);\n            unsigned int numOfEdges = findNumOfEdges(graph.vertexNum, vertexDegrees_s, vertexDegrees_s2, left, right);\n            endTime(NUM_EDGES,&blockCounters);\n            \n            bool numEdgesCondition = false;\n            if(numOfEdges>=square((int)minimum_s-(int)numDeletedVertices-1)+1) {\n                numEdgesCondition = true;\n            }\n            __syncthreads();\n\n            if(threadIdx.x == 0) {\n                minimum_s = atomicOr(minimum,0);\n            }\n            __syncthreads();\n            \n            if(numDeletedVertices >= minimum_s || numEdgesCondition) {\n                minExceeded = true;\n            } else {\n\n                unsigned int maxVertex = 0;\n                INT_OR_SHORT maxDegree = 0;\n                \n                startTime(MAX_DEGREE,&blockCounters);\n                findMaxDegree(graph.vertexNum, &maxVertex, &maxDegree, vertexDegrees_s, vertexDegrees_s2, left, right);\n                endTime(MAX_DEGREE,&blockCounters);\n\n                if(maxDegree<1){\n                    vcFound = true;\n                    if(threadIdx.x == 0) {\n                        atomicMin(minimum, numDeletedVertices);\n                    }\n                    __syncthreads();\n                } else {\n                    unsigned int goLeft = !((path >> k) & 1u);\n                    if(goLeft){\n                        \n                        if (threadIdx.x == 0){\n                            vertexDegrees_s[maxVertex] = -1;\n                        }\n                        ++numDeletedVertices;\n                        __syncthreads();\n\n                        for(unsigned int edge = graph.srcPtr[maxVertex] + threadIdx.x; edge < graph.srcPtr[maxVertex + 1]; edge+=blockDim.x) {\n                            unsigned int neighbor = graph.dst[edge];\n                            if(vertexDegrees_s[neighbor] != -1){\n                                #if USE_SHORT_DEGREE\n                                atomicSubShort(&vertexDegrees_s[neighbor], 1);\n                                #else\n                                atomicSub(&vertexDegrees_s[neighbor], 1);\n                                #endif\n                            }\n                        }\n\n                    } else {// Delete Neighbors of maxVertex\n                        numDeletedVertices += maxDegree;\n\n                        for(unsigned int edge = graph.srcPtr[maxVertex] + threadIdx.x; edge < graph.srcPtr[maxVertex + 1]; edge += blockDim.x) {\n                            unsigned int neighbor = graph.dst[edge];\n                            if(vertexDegrees_s[neighbor] != -1) {\n                                for(unsigned int neighborEdge = graph.srcPtr[neighbor]; neighborEdge < graph.srcPtr[neighbor + 1]; ++neighborEdge) {\n                                    unsigned int neighborOfNeighbor = graph.dst[neighborEdge];\n                                    if(vertexDegrees_s[neighborOfNeighbor] != -1) {\n                                        #if USE_SHORT_DEGREE\n                                        atomicSubShort(&vertexDegrees_s[neighborOfNeighbor], 1);\n                                        #else\n                                        atomicSub(&vertexDegrees_s[neighborOfNeighbor], 1);\n                                        #endif\n                                    }\n                                }\n                            }\n                        }\n                        __syncthreads();\n\n                        for(unsigned int edge = graph.srcPtr[maxVertex] + threadIdx.x; edge < graph.srcPtr[maxVertex + 1] ; edge += blockDim.x) {\n                            unsigned int neighbor = graph.dst[edge];\n                            vertexDegrees_s[neighbor] = -1;\n                        }\n                    }\n                }\n\n            }\n            __syncthreads();\n        }\n\n        // Each block its at it\'s required level which is at most Root depth\n        if(!vcFound && !minExceeded) {\n            unsigned int stackSize = (stacks.minimum + 1);\n            volatile INT_OR_SHORT * stackVertexDegrees = &stacks.stacks[(long long)blockIdx.x * (long long)stackSize * (long long)graph.vertexNum];\n            volatile unsigned int * stackNumDeletedVertices = &stacks.stacksNumDeletedVertices[blockIdx.x * stackSize];\n            volatile int * stackArrayIndex = &stacks.stackArrayIndex[blockIdx.x * stackSize];\n            volatile bool * stackCompNode = &stacks.stackCompNode[blockIdx.x * stackSize];\n\n            int arrayIndex = -1;\n            bool compNode = 0;\n            int stackTop = -1;\n\n            // go into while loop \n            bool popNextItr = false;\n            if (threadIdx.x == 0) {\n                s_current_depth = startingDepth;\n                s_last_branched_degree = 9999;\n            }\n            __syncthreads();\n            \n            do{\n                if(popNextItr) {\n                    startTime(POP_FROM_STACK,&blockCounters);\n                    popStack(graph.vertexNum, vertexDegrees_s, &numDeletedVertices, &arrayIndex, &compNode, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, stackCompNode, &stackTop, true);\n                    endTime(POP_FROM_STACK,&blockCounters);\n\n                    if (threadIdx.x == 0) {\n                        s_current_depth = startingDepth + stackTop + 1;\n                        s_last_branched_degree = 9999;\n                    }\n\n                    #if USE_COUNTERS\n                        if (threadIdx.x==0){\n                            atomicAdd(&NODES_PER_SM[sm_id],1);\n                        }\n                    #endif\n                }\n                __syncthreads();\n                \n                if(threadIdx.x == 0) {\n                    left = graph.vertexNum-1;\n                    right = 0;\n                }\n                __syncthreads();\n                \n                if (findBound) {\n                    startTime(FIND_BOUND,&blockCounters);\n                    findBoundaries(vertexDegrees_s, &left, &right, graph.vertexNum);\n                    endTime(FIND_BOUND,&blockCounters);\n                }\n\n                //reduction rule\n                __shared__ unsigned int minimum_s;\n                if(threadIdx.x == 0) {\n                    if (arrayIndex>-1) {\n                        #if USE_SHORT_DEGREE\n                        minimum_s = readIntFromShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum]);\n                        #else\n                        minimum_s = stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum];\n                        #endif\n                    } else {\n                        minimum_s = atomicOr(minimum,0);\n                    }\n                    \n                    if (right == 0) {\n                        left = 0;\n                        right = graph.vertexNum-1;\n                    }\n                }\n                __syncthreads();\n        \n                unsigned int iterationCounter = 0, numDeletedVerticesLeaf, numDeletedVerticesTriangle, numDeletedVerticesHighDegree;\n                do {\n                    startTime(LEAF_REDUCTION,&blockCounters);\n                    numDeletedVerticesLeaf = leafReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, left, right);\n                    endTime(LEAF_REDUCTION,&blockCounters);\n                    numDeletedVertices += numDeletedVerticesLeaf;\n                    if(iterationCounter==0 || numDeletedVerticesLeaf>0 || numDeletedVerticesHighDegree>0) {\n                        startTime(TRIANGLE_REDUCTION,&blockCounters);\n                        numDeletedVerticesTriangle = triangleReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, left, right);\n                        endTime(TRIANGLE_REDUCTION,&blockCounters);\n                        numDeletedVertices += numDeletedVerticesTriangle;\n                    } else {\n                        numDeletedVerticesTriangle = 0;\n                    }\n                    if(iterationCounter==0 || numDeletedVerticesLeaf>0 || numDeletedVerticesTriangle>0) {\n                        startTime(HIGH_DEGREE_REDUCTION,&blockCounters);\n                        numDeletedVerticesHighDegree = highDegreeReductionRule(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2,numDeletedVertices,minimum_s, left, right);\n                        endTime(HIGH_DEGREE_REDUCTION,&blockCounters);\n                        numDeletedVertices += numDeletedVerticesHighDegree;\n                    } else {\n                        numDeletedVerticesHighDegree = 0;\n                    }\n                    ++iterationCounter;\n                } while(numDeletedVerticesTriangle>0 || numDeletedVerticesHighDegree>0);\n\n                startTime(NUM_EDGES,&blockCounters);\n                unsigned int numOfEdges = findNumOfEdges(graph.vertexNum, vertexDegrees_s, vertexDegrees_s2, left, right);\n                endTime(NUM_EDGES,&blockCounters);\n\n                bool numEdgesCondition = false;\n                if(numOfEdges>=square((int)minimum_s-(int)numDeletedVertices-1)+1) {\n                    numEdgesCondition = true;\n                }\n                __syncthreads();\n        \n                if(threadIdx.x == 0) {\n                    if (arrayIndex>-1) {\n                        #if USE_SHORT_DEGREE\n                        minimum_s = readIntFromShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum]);\n                        #else\n                        minimum_s = stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum];\n                        #endif\n                    } else {\n                        minimum_s = atomicOr(minimum,0);\n                    }\n                }\n                __syncthreads();\n\n                if(numDeletedVertices>=minimum_s || numEdgesCondition) {\n                    if (threadIdx.x == 0 && arrayIndex>-1 && compNode) {\n                        #if USE_SHORT_DEGREE\n                        int counter = readIntFromShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3]);\n                        saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3], counter-1);\n                        stackNumDeletedVertices[arrayIndex] += readIntFromShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum]);\n                        if (arrayIndex != stackTop) {\n                            unsigned int minim = *minimum;\n                            if (stackArrayIndex[arrayIndex]>-1) {\n                                minim = readIntFromShortArray(&stackVertexDegrees[(long long)stackArrayIndex[arrayIndex]*(long long)graph.vertexNum]);\n                            }\n                            if (minim>stackNumDeletedVertices[arrayIndex]+counter) {\n                                    saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum], minim-stackNumDeletedVertices[arrayIndex]-counter+1);\n                            } else {\n                                stackNumDeletedVertices[arrayIndex] += counter;\n                                saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum], 0);\n                            } \n                        } else {\n                            saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum], 0);\n                            saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3], 0);\n                        }\n                        #else\n                        int counter = stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3];\n                        stackVertexDegrees[arrayIndex*graph.vertexNum+3] = counter-1;\n                        stackNumDeletedVertices[arrayIndex] += stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum];\n                        if (arrayIndex != stackTop) {\n                            unsigned int minim = *minimum;\n                            if (stackArrayIndex[arrayIndex]>-1) {\n                                minim = stackVertexDegrees[(long long)stackArrayIndex[arrayIndex]*(long long)graph.vertexNum];\n                            }\n                            if (minim>stackNumDeletedVertices[arrayIndex]+counter) {\n                                    stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum] = minim-stackNumDeletedVertices[arrayIndex]-counter+1;\n                            } else {\n                                stackNumDeletedVertices[arrayIndex] += counter;\n                                stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum] = 0;\n                            } \n                        } else {\n                            stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum] = 0;\n                            stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3] = 0;\n                        }\n                        #endif\n                    }\n                    popNextItr = true;\n                    __syncthreads();\n                    if (threadIdx.x == 0) {\n                        run_cc_scan = (s_current_depth == 0) || \n                                      (s_last_branched_degree >= LAZY_TAU) || \n                                      (s_current_depth % LAZY_STRIDE == 0);\n                    }\n                    __syncthreads();\n\n                    unsigned int isComponents = 1;\n                    if (run_cc_scan) {\n                        #if USE_COUNTERS\n                        if (threadIdx.x == 0) {\n                            blockCounters.totalTime[TOTAL_BFS_SCANS]++;\n                        }\n                        #endif\n                        startTime(COMP_SEARCH,&blockCounters);\n                        isComponents = bfsCC_StackOnly(graph.vertexNum, vertexDegrees_s, graph, vertexDegrees_s2, arrayIndex, compNode, minimum_s, &numDeletedVertices, &blockCounters, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, stackCompNode, &stackTop, stackSize, left, right);\n                        endTime(COMP_SEARCH,&blockCounters);\n                        #if USE_COUNTERS\n                        if (threadIdx.x == 0 && isComponents == 1) {\n                            blockCounters.totalTime[UNPRODUCTIVE_BFS_SCANS]++;\n                        }\n                        #endif\n                    } else {\n                        isComponents = 1;\n                    }\n                    __syncthreads();\n                    \n                    unsigned int maxVertex;\n                    INT_OR_SHORT maxDegree;\n                    if (isComponents>1) {\n                        popNextItr = true;\n                        __syncthreads();\n                        \n                    } else if (isComponents==0) {\n                        if(threadIdx.x == 0) {\n                            if (arrayIndex>-1) {\n                                #if USE_SHORT_DEGREE\n                                if (numDeletedVertices<readIntFromShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum])) {\n                                    saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum], numDeletedVertices);\n                                }\n                                if (compNode) {\n                                    int counter = readIntFromShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3]);\n                                    saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3], counter-1);\n                                    stackNumDeletedVertices[arrayIndex] += readIntFromShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum]);\n                                    if (arrayIndex != stackTop) {\n                                        unsigned int minim = *minimum;\n                                        if (stackArrayIndex[arrayIndex]>-1) {\n                                            minim = readIntFromShortArray(&stackVertexDegrees[(long long)stackArrayIndex[arrayIndex]*(long long)graph.vertexNum]);\n                                        }\n                                        if (minim>stackNumDeletedVertices[arrayIndex]+counter) {\n                                                saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum], minim-stackNumDeletedVertices[arrayIndex]-counter+1);\n                                        } else {\n                                            stackNumDeletedVertices[arrayIndex] += counter;\n                                            saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum], 0);\n                                        }\n                                    } else {\n                                        saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum], 0);\n                                        saveIntInShortArray(&stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3], 0);\n                                    }\n                                }\n                                #else\n                                if (numDeletedVertices<stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum]) {\n                                    stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum] = numDeletedVertices;\n                                }\n                                if (compNode) {\n                                    int counter = stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3];\n                                    stackVertexDegrees[arrayIndex*graph.vertexNum+3] = counter-1;\n                                    stackNumDeletedVertices[arrayIndex] += stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum];\n                                    if (arrayIndex != stackTop) {\n                                        unsigned int minim = *minimum;\n                                        if (stackArrayIndex[arrayIndex]>-1) {\n                                            minim = stackVertexDegrees[(long long)stackArrayIndex[arrayIndex]*(long long)graph.vertexNum];\n                                        }\n                                        if (minim>stackNumDeletedVertices[arrayIndex]+counter) {\n                                                stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum] = minim-stackNumDeletedVertices[arrayIndex]-counter+1;\n                                        } else {\n                                            stackNumDeletedVertices[arrayIndex] += counter;\n                                            stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum] = 0;\n                                        }\n                                    } else {\n                                        stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum] = 0;\n                                        stackVertexDegrees[(long long)arrayIndex*(long long)graph.vertexNum+(long long)3] = 0;\n                                    }\n                                }\n                                #endif\n                            } else {\n                                atomicMin(minimum, numDeletedVertices);\n                            }\n                        }\n                        popNextItr = true;\n                        __syncthreads();\n                    } else {\n                    \n                        // Find max degree\n                        startTime(MAX_DEGREE,&blockCounters);\n                        findMaxDegree(graph.vertexNum, &maxVertex, &maxDegree, vertexDegrees_s, vertexDegrees_s2, left, right);\n                        endTime(MAX_DEGREE,&blockCounters);\n\n                        startTime(PREPARE_RIGHT_CHILD,&blockCounters);\n                        deleteNeighborsOfMaxDegreeVertex(graph,vertexDegrees_s, &numDeletedVertices, vertexDegrees_s2, &numDeletedVertices2, maxDegree, maxVertex);\n                        endTime(PREPARE_RIGHT_CHILD,&blockCounters);\n\n                        startTime(PUSH_TO_STACK,&blockCounters);\n                        pushStack(graph.vertexNum, vertexDegrees_s2, &numDeletedVertices2, arrayIndex, compNode, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, stackCompNode, &stackTop, true);\n                        endTime(PUSH_TO_STACK,&blockCounters);\n                        maxDepth(stackTop, &blockCounters);\n                        __syncthreads();\n\n                        startTime(PREPARE_LEFT_CHILD,&blockCounters);\n                        deleteMaxDegreeVertex(graph, vertexDegrees_s, &numDeletedVertices, maxVertex);\n                        endTime(PREPARE_LEFT_CHILD,&blockCounters);\n\n                        #if USE_COUNTERS\n                            if (threadIdx.x==0){\n                                atomicAdd(&NODES_PER_SM[sm_id],1);\n                            }\n                        #endif\n\n                        compNode = 0;\n                        popNextItr = false;\n                        if (threadIdx.x == 0) {\n                            s_current_depth++;\n                            s_last_branched_degree = maxDegree;\n                        }\n                        __syncthreads();\n                    }\n                }\n                __syncthreads();\n\n            } while(stackTop != -1);\n        }\n    }while(true);\n\n    #if USE_COUNTERS\n    if(threadIdx.x == 0) {\n        counters[blockIdx.x] = blockCounters;\n    }\n    #endif\n}\n\n#undef INT_OR_SHORT\n#undef FUNCTION_NAME(base)\n#undef SHARED_MEM_TYPE(base)')

with open("solver_variant_d_full/src/helperFunctions.cuh", "w") as f:
    f.write('#if (!defined(HELPFUNCSHORT_H) && USE_SHORT_DEGREE) || (!defined(HELPFUNCINT_H) && !USE_SHORT_DEGREE)\n\n#include "config.h"\n#include "BWDWorkList.cuh"\n#include "stack.cuh"\n\n#if USE_SHORT_DEGREE\n#define HELPFUNCSHORT_H\n#define INT_OR_SHORT short\n#define ATOMIC_TYPE(base) base##Short\n\n__device__ short atomicSubShort(short* address, short val) {\n    // Convert the short* address to a char* for byte-level operations\n    char* addr_as_char = (char*)address;\n\n    // Compute the base address aligned to 4 bytes (32 bits)\n    char* base_addr_char = addr_as_char - ((size_t)addr_as_char % 4);\n    int* base_addr = (int*)base_addr_char;\n\n    // Calculate the offset within the 32-bit word (0 or 2)\n    size_t offset = (size_t)(addr_as_char - base_addr_char);\n\n    int old = *base_addr;\n    int assumed;\n    int newval;\n    short old_short;\n\n    do {\n        assumed = old;\n\n        // Extract the lower and upper 16 bits\n        short low = (short)(old & 0xFFFF);\n        short high = (short)((old >> 16) & 0xFFFF);\n\n        if (offset == 0) {\n            // We are working with the lower 16 bits\n            old_short = low;\n            low = old_short - val;\n        } else {\n            // We are working with the upper 16 bits\n            old_short = high;\n            high = old_short - val;\n        }\n\n        // Reconstruct the 32-bit integer\n        newval = ((int)(unsigned short)high << 16) | (unsigned short)(low);\n\n        // Attempt to atomically update the 32-bit word\n        old = atomicCAS(base_addr, assumed, newval);\n    } while (assumed != old);\n\n    return old_short;\n}\n\n__device__ void saveIntInShortArray(volatile short* arr, int value) {\n    arr[0] = value & 0xFFFF;          // Lower 16 bits\n    arr[1] = (value >> 16) & 0xFFFF;  // Upper 16 bits\n}\n\n__device__ void saveIntInShortArray(short* arr, int value) {\n    arr[0] = value & 0xFFFF;          // Lower 16 bits\n    arr[1] = (value >> 16) & 0xFFFF;  // Upper 16 bits\n}\n\n__device__ int readIntFromShortArray(volatile short* arr) {\n    int value = (static_cast<int>(arr[1]) << 16) | (static_cast<unsigned short>(arr[0]));\n    return value;\n}\n\n__device__ int readIntFromShortArray(short* arr) {\n    int value = (static_cast<int>(arr[1]) << 16) | (static_cast<unsigned short>(arr[0]));\n    return value;\n}\n\n__device__ void AddIntInShortArray(volatile short* arr, int value) {\n    int currentValue = readIntFromShortArray(arr);\n    int newValue = currentValue + value;\n    saveIntInShortArray(arr, newValue);\n}\n\n__device__ void AddIntInShortArray(short* arr, int value) {\n    int currentValue = readIntFromShortArray(arr);\n    int newValue = currentValue + value;\n    saveIntInShortArray(arr, newValue);\n}\n\n\n#else\n#define HELPFUNCINT_H\n#define INT_OR_SHORT int\n#define ATOMIC_TYPE(base) base\n\n__device__ long long int square(int num) {\n    return num*num;\n}\n\n__device__ bool binarySearch(unsigned int * arr,  int l, int r, unsigned int x) {\n\n   while (l <= r) {\n        int m = l + (r - l) / 2;\n        if (arr[m] == x)\n            return  true;\n  \n        if (arr[m] < x)\n            l = m + 1;\n  \n        else\n            r = m - 1;\n    }\n    return false;\n}\n\n#endif\n\n__device__ void findBoundaries(INT_OR_SHORT* vertexDegrees_s, unsigned int* left, unsigned int* right, int vertexNum){\n    for(unsigned int vertex = threadIdx.x; vertex < vertexNum; vertex += blockDim.x){\n        if(vertexDegrees_s[vertex] > 0){\n            atomicMin(left,vertex);\n            break;\n        }\n    }\n    for(int vertex = vertexNum-(int)(threadIdx.x)-1; vertex > -1; vertex -= (int)blockDim.x){\n        if(vertexDegrees_s[vertex] > 0){\n            atomicMax(right,vertex);\n            break;\n        }\n    }\n    __syncthreads();\n}\n\n__device__ void deleteNeighborsOfMaxDegreeVertex(CSRGraph graph, INT_OR_SHORT* vertexDegrees_s, unsigned int* numDeletedVertices, INT_OR_SHORT* vertexDegrees_s2, unsigned int* numDeletedVertices2, INT_OR_SHORT maxDegree, unsigned int maxVertex) {\n\n    *numDeletedVertices2 = *numDeletedVertices;\n    for(unsigned int vertex = threadIdx.x; vertex<graph.vertexNum; vertex+=blockDim.x){\n        vertexDegrees_s2[vertex] = vertexDegrees_s[vertex];\n    }\n    __syncthreads();\n\n    for(unsigned int edge = graph.srcPtr[maxVertex] + threadIdx.x; edge < graph.srcPtr[maxVertex + 1]; edge+=blockDim.x) { // Delete Neighbors of maxVertex\n        unsigned int neighbor = graph.dst[edge];\n        if (vertexDegrees_s2[neighbor] > 0){\n            for(unsigned int neighborEdge = graph.srcPtr[neighbor]; neighborEdge < graph.srcPtr[neighbor + 1]; ++neighborEdge) {\n                unsigned int neighborOfNeighbor = graph.dst[neighborEdge];\n                if(vertexDegrees_s2[neighborOfNeighbor] > 0) {\n                    ATOMIC_TYPE(atomicSub)(&vertexDegrees_s2[neighborOfNeighbor], 1);\n                }\n            }\n        }\n    }\n    \n    *numDeletedVertices2 += maxDegree;\n    __syncthreads();\n\n    for(unsigned int edge = graph.srcPtr[maxVertex] + threadIdx.x; edge < graph.srcPtr[maxVertex + 1] ; edge += blockDim.x) {\n        unsigned int neighbor = graph.dst[edge];\n        vertexDegrees_s2[neighbor] = -1;\n    }\n    __syncthreads();\n}\n\n__device__ void deleteMaxDegreeVertex(CSRGraph graph, INT_OR_SHORT* vertexDegrees_s, unsigned int* numDeletedVertices, unsigned int maxVertex) {\n\n    if(threadIdx.x == 0){\n        vertexDegrees_s[maxVertex] = -1;\n    }\n    ++(*numDeletedVertices);\n\n    __syncthreads(); \n\n    for(unsigned int edge = graph.srcPtr[maxVertex] + threadIdx.x; edge < graph.srcPtr[maxVertex + 1]; edge += blockDim.x) {\n        unsigned int neighbor = graph.dst[edge];\n        if(vertexDegrees_s[neighbor] > 0) {\n            --vertexDegrees_s[neighbor];\n        }\n    }\n    __syncthreads();\n}\n\n__device__ unsigned int leafReductionRule(unsigned int vertexNum, INT_OR_SHORT *vertexDegrees_s, CSRGraph graph, INT_OR_SHORT* shared_mem, unsigned int left, unsigned int right) {\n\n    __shared__ unsigned int numberDeleted;\n    __shared__ bool graphHasChanged;\n\n    volatile INT_OR_SHORT * markedForDeletion = shared_mem;\n    if (threadIdx.x==0){\n        numberDeleted = 0;\n    }\n    unsigned int start = threadIdx.x + left;\n    do{\n        volatile INT_OR_SHORT * vertexDegrees_v = vertexDegrees_s;\n        __syncthreads();\n                \n        for (unsigned int vertex = start ; vertex < right+1; vertex += blockDim.x){\n            markedForDeletion[vertex] = 0;\n        }\n        if (threadIdx.x==0){\n            graphHasChanged = false;\n        }\n        \n        __syncthreads();\n\n        for (unsigned int vertex = start ; vertex < right+1; vertex+=blockDim.x){\n            INT_OR_SHORT degree = vertexDegrees_v[vertex];\n            if (degree == 1){\n                for(unsigned int edge = graph.srcPtr[vertex] ; edge < graph.srcPtr[vertex + 1]; ++edge) {\n                    unsigned int neighbor = graph.dst[edge];\n                    INT_OR_SHORT neighborDegree = vertexDegrees_v[neighbor];\n                    if(neighborDegree>0){\n                        if(neighborDegree > 1 || (neighborDegree == 1 && neighbor<vertex)) {\n                            markedForDeletion[neighbor]=1;\n                        }\n                        break;\n                    }\n                }\n            }\n        }\n        \n        __syncthreads();\n        \n        for (unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x){\n            if(markedForDeletion[vertex]){ \n                graphHasChanged = true;\n                vertexDegrees_v[vertex] = -1;\n                atomicAdd(&numberDeleted,1);\n            }\n        }\n        \n        __syncthreads();\n        \n        for (unsigned int vertex = start ; vertex < right+1; vertex+=blockDim.x){\n            if(markedForDeletion[vertex]){ \n                for(unsigned int edge = graph.srcPtr[vertex]; edge < graph.srcPtr[vertex + 1]; edge++){\n                    unsigned int neighbor = graph.dst[edge];\n                    if (vertexDegrees_v[neighbor] > 0){\n                        ATOMIC_TYPE(atomicSub)(&vertexDegrees_s[neighbor],1);\n                    }\n                }\n            }\n        }\n    \n    }while(graphHasChanged);\n\n    __syncthreads();\n\n    return numberDeleted;\n}\n\n__device__ unsigned int cycleReductionRule(unsigned int vertexNum, INT_OR_SHORT *vertexDegrees_s, CSRGraph graph, INT_OR_SHORT * shared_mem, unsigned int left, unsigned int right) {\n    __shared__ unsigned int numberDeleted;\n    if(threadIdx.x == 0) {\n        numberDeleted = 0;\n    }\n    volatile INT_OR_SHORT * markedForDeletion = shared_mem;\n    INT_OR_SHORT * vertexDegrees_v = vertexDegrees_s;\n    unsigned int start = threadIdx.x + left;\n    for (unsigned int vertex = start; vertex < right+1; vertex += blockDim.x){\n        markedForDeletion[vertex] = 0;\n    }\n    __syncthreads();\n\n    for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n        INT_OR_SHORT degree = vertexDegrees_v[vertex];\n        if(degree != 2) {\n            continue;\n        }\n        bool secondFound = false;\n        bool big_brother = false;\n        unsigned int last;\n        unsigned int second;\n        for(unsigned int edge = graph.srcPtr[vertex]; edge<graph.srcPtr[vertex+1]; ++edge) {\n            unsigned int neigh = graph.dst[edge];\n            if(vertexDegrees_v[neigh]>0) {\n                if(neigh<vertex) {\n                    big_brother = true;\n                    break;\n                }\n                if(secondFound) {\n                    last = neigh;\n                    break;\n                } else {\n                    secondFound = true;\n                    second = neigh;\n                }\n            }\n        }\n\n        if(big_brother) {\n            continue;\n        }\n        \n        if(vertexDegrees_v[last] != 2) {\n                continue;\n        }\n\n        unsigned int first = vertex;\n        bool isCycle = true;\n        bool done = false;\n\n        while(isCycle) {\n            if (vertexDegrees_v[second] != 2) {\n                break;\n            }\n            for(unsigned int edge=graph.srcPtr[second]; edge<graph.srcPtr[second+1]; ++edge) {\n                unsigned int neigh = graph.dst[edge];\n                if(neigh!=first && vertexDegrees_v[neigh]>0) {\n                    if(neigh<vertex || vertexDegrees_v[neigh] != 2) {\n                        isCycle = false;\n                        break;\n                    } else if (neigh!=last) {\n                        first = second;\n                        second = neigh;\n                        break;\n                    } else {\n                        done = true;\n                        break;\n                    }\n                }\n            }\n            if(done) {\n                markedForDeletion[vertex] = 1;\n                unsigned int pointer = vertex;\n                unsigned int prev = last;\n                unsigned int cycleSize = 1;\n                while(pointer!=last) {\n                    ++cycleSize;\n                    for(unsigned int e = graph.srcPtr[pointer]; e<graph.srcPtr[pointer+1]; ++e) {\n                        unsigned int n = graph.dst[e];\n                        if(n!=prev && vertexDegrees_v[n]>0) {\n                            markedForDeletion[n] = 1;\n                            prev = pointer;\n                            pointer = n;\n                            break;\n                        }\n                    }\n                }\n                atomicAdd(&numberDeleted,(cycleSize+1)/2);\n                // For Debugging Purposes\n                //printf("Cycle Size: %d\\n",cycleSize);\n                break;\n            }\n        }\n    }\n    __syncthreads();\n   \n    for (unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n        if(markedForDeletion[vertex]) { \n            vertexDegrees_v[vertex] = -1;\n        }\n    }\n\n   __syncthreads();\n   return numberDeleted;\n}\n\n__device__ unsigned int cliqueReductionRule(unsigned int vertexNum, INT_OR_SHORT* vertexDegrees_s, CSRGraph graph, INT_OR_SHORT * shared_mem, unsigned int left, unsigned int right) {\n    __shared__ unsigned int numberDeleted;\n    if(threadIdx.x == 0) {\n        numberDeleted = 0;\n    }\n    volatile INT_OR_SHORT * markedForDeletion = shared_mem;\n    volatile INT_OR_SHORT * vertexDegrees_v = vertexDegrees_s;\n    unsigned int start = threadIdx.x + left;\n    for (unsigned int vertex = start; vertex < right+1; vertex += blockDim.x){\n        markedForDeletion[vertex] = 0;\n    }\n    __syncthreads();\n    \n    for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n        INT_OR_SHORT degree = vertexDegrees_v[vertex];\n        unsigned int k = degree+1;\n        if(degree>2) {\n            bool inClique = true;\n            for(unsigned int edge = graph.srcPtr[vertex]; edge<graph.srcPtr[vertex+1]; ++edge) {\n                unsigned int neigh = graph.dst[edge];\n                if(vertexDegrees_v[neigh]>0) {\n                    if(neigh<vertex || vertexDegrees_v[neigh]!=k-1) {\n                        inClique = false;\n                        break;\n                    }\n                }\n            }\n            if(inClique) {\n                for(unsigned int edge = graph.srcPtr[vertex]; edge<graph.srcPtr[vertex+1]; ++edge) {\n                    unsigned int neigh = graph.dst[edge];\n\n                    if(vertexDegrees_v[neigh]>0) {\n                        for(unsigned int edgeN = graph.srcPtr[neigh]; edgeN<graph.srcPtr[neigh+1]; ++edgeN) {\n                            unsigned int neighN = graph.dst[edgeN];\n\n                            if(vertexDegrees_v[neighN]>0 && neighN!=vertex) {\n                                bool found = false;\n                                for(unsigned int edgeNN = graph.srcPtr[vertex]; edgeNN<graph.srcPtr[vertex+1]; ++edgeNN) {\n                                    unsigned int neighNN = graph.dst[edgeNN];\n                                    if(vertexDegrees_v[neighNN]>0 && neighN==neighNN) {\n                                        found  = true;\n                                        break;\n                                    }\n                                }\n                                if(!found) {\n                                    inClique = false;\n                                    break;\n                                }\n                            }\n                        }\n                        if(!inClique) {\n                            break;\n                        }\n                    }\n                }   \n            }\n            if(inClique) {\n                atomicAdd(&numberDeleted,k-1);\n                markedForDeletion[vertex] = 1;\n                for(unsigned int edge = graph.srcPtr[vertex]; edge<graph.srcPtr[vertex+1]; ++edge) {\n                    unsigned int neigh = graph.dst[edge];\n                    if (vertexDegrees_v[neigh]>0) {\n                        markedForDeletion[neigh] = 1;\n                    }\n                }\n                // For Debugging Purposes\n                //printf("Clique Size: %d\\n",k);\n            }\n        }\n    }\n    __syncthreads();\n\n    for (unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n        if(markedForDeletion[vertex]) { \n            vertexDegrees_v[vertex] = -1;\n        }\n    }\n\n   __syncthreads();\n   return numberDeleted;\n}\n\n__device__ unsigned int bfsCC_StackOnly(unsigned int vertexNum, INT_OR_SHORT* vertexDegrees_s, CSRGraph graph, INT_OR_SHORT* shared_mem, int arrayIndex, bool compNode, unsigned int minimum_s, unsigned int* numDeletedVertices, Counters* blockCounters, volatile INT_OR_SHORT * stackVertexDegrees, volatile unsigned int * stackNumDeletedVertices, volatile int * stackArrayIndex, volatile bool * stackCompNode, int* stackTop, unsigned int stackSize, unsigned int left, unsigned int right) {\n\n    __shared__ int num_components;\n    __shared__ int found_new_level;\n    __shared__ bool done;\n    __shared__ int select_comp;\n    __shared__ bool exceedFlag;\n    __shared__ int compSize;\n    __shared__ int not_Clique_Cycle;\n    __shared__ int compStartingVertexDegree;\n    __shared__ int s_degree_sum_i;\n    __shared__ int s_max_deg_i;\n    __shared__ int s_fast_path_type;\n    __shared__ int s_tree_cover;\n    unsigned int parentIndex;\n    volatile INT_OR_SHORT* components = shared_mem;\n    unsigned int start = threadIdx.x + left;\n    \n    if (threadIdx.x == 0) {\n        done = true;\n        num_components = 0;\n        exceedFlag = false;\n    }\n    __syncthreads();\n\n    for(unsigned int vertex = threadIdx.x; vertex < graph.vertexNum; vertex+=blockDim.x) {\n        if(vertexDegrees_s[vertex] > 0) {\n            components[vertex] = 0;\n            done = false;\n        } else {\n            components[vertex] = -1;\n        }\n    }\n    __syncthreads();\n    \n    if (done) {return 0;}\n\n    startTime(PUSH_TO_STACK, blockCounters);\n    pushStack(vertexNum, (INT_OR_SHORT*)components, numDeletedVertices, arrayIndex, compNode, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, stackCompNode, stackTop, true);\n    endTime(PUSH_TO_STACK, blockCounters);\n    maxDepth(*stackTop, blockCounters);\n    parentIndex = *stackTop;\n     __syncthreads();\n\n    do {\n        __syncthreads();\n        if(threadIdx.x == 0) {\n            select_comp = 0;\n            compSize = 1;\n            not_Clique_Cycle = 0;\n            ++num_components;\n            done = true;\n        }\n        __syncthreads();\n\n        for (unsigned int i = start; i<right+1 && !select_comp; i+=blockDim.x) {\n            if(components[i] == 0) {\n                if(atomicCAS(&select_comp,0,1) == 0) {\n                    components[i] = 1;\n                    compStartingVertexDegree = vertexDegrees_s[i];\n                }\n            }\n        }\n        __syncthreads();\n\n        do {\n            __syncthreads();\n            if (threadIdx.x == 0) {\n                found_new_level = 0;\n            }\n             __syncthreads();\n            for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n                if(components[vertex] == 0) {\n                    for(unsigned int edge = graph.srcPtr[vertex]; edge < graph.srcPtr[vertex+1]; ++edge) {\n                        unsigned int neigh = graph.dst[edge];\n                        if (vertexDegrees_s[neigh]>0) {\n                            if(components[neigh] == 1) {\n                                components[vertex] = 1;\n                                atomicAdd(&compSize,1);\n                                found_new_level = 1;\n                                if (vertexDegrees_s[vertex]!=compStartingVertexDegree) {\n                                    not_Clique_Cycle = 1;\n                                }\n                                break;\n                            }\n                        }\n                    }\n                }\n            }\n            __syncthreads();\n        } while (found_new_level);\n        \n        if (threadIdx.x == 0) {\n            s_degree_sum_i = 0;\n            s_max_deg_i = 0;\n            s_fast_path_type = 0;\n            s_tree_cover = 0;\n        }\n        __syncthreads();\n\n        int local_deg_sum = 0;\n        int local_max_deg = 0;\n        for (unsigned int v = start; v < right + 1; v += blockDim.x) {\n            if (components[v] == 1 && vertexDegrees_s[v] > 0) {\n                int d = vertexDegrees_s[v];\n                local_deg_sum += d;\n                if (d > local_max_deg) local_max_deg = d;\n            }\n        }\n        atomicAdd(&s_degree_sum_i, local_deg_sum);\n        atomicMax(&s_max_deg_i, local_max_deg);\n        __syncthreads();\n\n        if (threadIdx.x == 0) {\n            int n_i = compSize;\n            int m_i = s_degree_sum_i / 2;\n            if (n_i >= 3 && m_i == n_i && s_max_deg_i == 2) {\n                s_fast_path_type = 1; // Simple Cycle\n            } else if (n_i >= 1 && m_i == n_i - 1) {\n                s_fast_path_type = 2; // Acyclic Tree / Star\n            } else if (n_i >= 2 && compStartingVertexDegree == n_i - 1 && !not_Clique_Cycle) {\n                s_fast_path_type = 3; // Clique\n            }\n        }\n        __syncthreads();\n\n        if (s_fast_path_type == 1) {\n            int cycle_cover = (compSize + 1) / 2;\n            *numDeletedVertices += cycle_cover;\n            if (threadIdx.x == 0) {\n                #if USE_COUNTERS\n                blockCounters->totalTime[CYCLE_FAST_PATH_HITS]++;\n                #endif\n                stackNumDeletedVertices[parentIndex] += cycle_cover;\n                --num_components;\n            }\n        } else if (s_fast_path_type == 2) {\n            if (compSize == 1) {\n                if (threadIdx.x == 0) {\n                    --num_components;\n                }\n            } else if (compSize == 2 || s_max_deg_i == compSize - 1) {\n                *numDeletedVertices += 1;\n                if (threadIdx.x == 0) {\n                    #if USE_COUNTERS\n                    blockCounters->totalTime[TREE_FAST_PATH_HITS]++;\n                    #endif\n                    stackNumDeletedVertices[parentIndex] += 1;\n                    --num_components;\n                }\n            } else {\n                __shared__ int selected_leaf;\n                __shared__ int parent_v;\n                while (true) {\n                    if (threadIdx.x == 0) {\n                        selected_leaf = -1;\n                        parent_v = -1;\n                    }\n                    __syncthreads();\n\n                    for (unsigned int v = start; v < right + 1; v += blockDim.x) {\n                        if (components[v] == 1 && vertexDegrees_s[v] == 1) {\n                            atomicMax(&selected_leaf, (int)v);\n                            break;\n                        }\n                    }\n                    __syncthreads();\n\n                    if (selected_leaf == -1) break;\n\n                    int u = selected_leaf;\n                    for (unsigned int e = graph.srcPtr[u] + threadIdx.x; e < graph.srcPtr[u + 1]; e += blockDim.x) {\n                        unsigned int nb = graph.dst[e];\n                        if (components[nb] == 1 && vertexDegrees_s[nb] > 0) {\n                            parent_v = (int)nb;\n                        }\n                    }\n                    __syncthreads();\n\n                    if (parent_v != -1) {\n                        int p = parent_v;\n                        for (unsigned int e = graph.srcPtr[p] + threadIdx.x; e < graph.srcPtr[p + 1]; e += blockDim.x) {\n                            unsigned int nb = graph.dst[e];\n                            if (components[nb] == 1 && vertexDegrees_s[nb] > 0) {\n                                atomicSub((int*)&vertexDegrees_s[nb], 1);\n                            }\n                        }\n                        __syncthreads();\n                        if (threadIdx.x == 0) {\n                            vertexDegrees_s[p] = 0;\n                            s_tree_cover += 1;\n                        }\n                        __syncthreads();\n                    } else {\n                        if (threadIdx.x == 0) vertexDegrees_s[u] = 0;\n                        __syncthreads();\n                    }\n                }\n\n                *numDeletedVertices += s_tree_cover;\n                if (threadIdx.x == 0) {\n                    #if USE_COUNTERS\n                    blockCounters->totalTime[TREE_FAST_PATH_HITS]++;\n                    #endif\n                    stackNumDeletedVertices[parentIndex] += s_tree_cover;\n                    --num_components;\n                }\n            }\n        } else if (s_fast_path_type == 3) {\n            int clique_cover = compSize - 1;\n            *numDeletedVertices += clique_cover;\n            if (threadIdx.x == 0) {\n                stackNumDeletedVertices[parentIndex] += clique_cover;\n                --num_components;\n            }\n        }\n        __syncthreads();\n\n        for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n            if(components[vertex] == 0) {\n                done = false;\n                break;\n            }\n        }\n        __syncthreads();\n        \n        if (s_fast_path_type > 0) {\n            for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n                if(components[vertex] > 0){\n                    components[vertex] = -1;\n                    vertexDegrees_s[vertex] = -1;\n                }\n            }\n            __syncthreads();\n                \n            if (!done) {\n                continue;\n            }\n            __syncthreads();\n\n            if (num_components==0) {\n                --(*stackTop);\n                return 0;\n            } else if (num_components==1) {\n                (*stackTop) -= 2;\n                return num_components;\n            }\n            __syncthreads();\n\n            break;\n        }\n        __syncthreads();\n        \n        if(done && num_components == 1) {\n            --(*stackTop);\n            return num_components;\n        }\n        \n        for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n            if(components[vertex] == 1) {\n                components[vertex] = vertexDegrees_s[vertex];\n            }\n        }\n        __syncthreads();\n         \n        unsigned int numDeletedVertices2 = 0;\n        startTime(PUSH_TO_STACK, blockCounters);\n        pushStack(vertexNum, (INT_OR_SHORT*)components, &numDeletedVertices2, parentIndex, 1, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, stackCompNode, stackTop, true);\n        endTime(PUSH_TO_STACK,blockCounters);\n        maxDepth(*stackTop, blockCounters);\n        __syncthreads();\n        \n        if (threadIdx.x==0 && *stackTop==stackSize-1) {\n            exceedFlag = true;\n        }\n        __syncthreads();\n        \n        if (exceedFlag) {\n            *stackTop = parentIndex-1;\n            *numDeletedVertices = minimum_s;\n            return 0; //no need for this node\n        }\n\n        for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x){\n            if(components[vertex] > 0){\n                components[vertex] = -1;\n            }\n        }\n        __syncthreads();\n\n    } while(!done);\n    __syncthreads();\n    \n    if (threadIdx.x==0) {\n        done = false;\n        #if USE_SHORT_DEGREE\n        saveIntInShortArray(&stackVertexDegrees[(long long)parentIndex*(long long)graph.vertexNum+(long long)3], num_components-1);\n        #else\n        stackVertexDegrees[(long long)parentIndex*(long long)graph.vertexNum+(long long)3] = num_components-1;\n        #endif\n        if (minimum_s>*numDeletedVertices+num_components) {\n            #if USE_SHORT_DEGREE\n            saveIntInShortArray(&stackVertexDegrees[(long long)parentIndex*(long long)graph.vertexNum], minimum_s-*numDeletedVertices-num_components+1);\n            #else\n            stackVertexDegrees[(long long)parentIndex*(long long)graph.vertexNum] = minimum_s-*numDeletedVertices-num_components+1;\n            #endif\n        } else {\n            done = true;\n        }\n    }\n    __syncthreads();\n    \n    if (done) {\n        *stackTop = parentIndex-1;\n        *numDeletedVertices = minimum_s;\n        return 0; //no need for this node\n    }\n    __syncthreads();\n\n    return num_components;\n}\n\n__device__ unsigned int bfsCC_param_StackOnly(unsigned int vertexNum, INT_OR_SHORT* vertexDegrees_s, CSRGraph graph, INT_OR_SHORT* shared_mem, int arrayIndex, bool compNode, unsigned int minimum_s, unsigned int* numDeletedVertices, Counters* blockCounters, volatile INT_OR_SHORT * stackVertexDegrees, volatile unsigned int * stackNumDeletedVertices, volatile int * stackArrayIndex, volatile bool * stackCompNode, int* stackTop, unsigned int stackSize, unsigned int * k, unsigned int left, unsigned int right) {\n\n    __shared__ int num_components;\n    __shared__ int found_new_level;\n    __shared__ bool done;\n    __shared__ int select_comp;\n    __shared__ bool exceedFlag;\n    __shared__ int compSize;\n    __shared__ int not_Clique_Cycle;\n    __shared__ int compStartingVertexDegree;\n    unsigned int parentIndex;\n    volatile INT_OR_SHORT* components = shared_mem;\n    unsigned int start = threadIdx.x + left;\n    \n    if (threadIdx.x == 0) {\n        done = true;\n        num_components = 0;\n        exceedFlag = false;\n    }\n    __syncthreads();\n\n    for(unsigned int vertex = threadIdx.x; vertex < graph.vertexNum; vertex+=blockDim.x) {\n        if(vertexDegrees_s[vertex] > 0) {\n            components[vertex] = 0;\n            done = false;\n        } else {\n            components[vertex] = -1;\n        }\n    }\n    __syncthreads();\n    \n    if (done) {return 0;}\n\n    startTime(PUSH_TO_STACK, blockCounters);\n    pushStack(vertexNum, (INT_OR_SHORT*)components, numDeletedVertices, arrayIndex, compNode, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, stackCompNode, stackTop, true);\n    endTime(PUSH_TO_STACK, blockCounters);\n    maxDepth(*stackTop, blockCounters);\n    parentIndex = *stackTop;\n     __syncthreads();\n\n    do {\n        __syncthreads();\n        if(threadIdx.x == 0) {\n            select_comp = 0;\n            compSize = 1;\n            not_Clique_Cycle = 0;\n            ++num_components;\n            done = true;\n        }\n        __syncthreads();\n\n        for (unsigned int i = start; i<right+1 && !select_comp; i+=blockDim.x) {\n            if(components[i] == 0) {\n                if(atomicCAS(&select_comp,0,1) == 0) {\n                    components[i] = 1;\n                    compStartingVertexDegree = vertexDegrees_s[i];\n                }\n            }\n        }\n        __syncthreads();\n\n        do {\n            __syncthreads();\n            if (threadIdx.x == 0) {\n                found_new_level = 0;\n            }\n             __syncthreads();\n            for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n                if(components[vertex] == 0) {\n                    for(unsigned int edge = graph.srcPtr[vertex]; edge < graph.srcPtr[vertex+1]; ++edge) {\n                        unsigned int neigh = graph.dst[edge];\n                        if (vertexDegrees_s[neigh]>0) {\n                            if(components[neigh] == 1) {\n                                components[vertex] = 1;\n                                atomicAdd(&compSize,1);\n                                found_new_level = 1;\n                                if (vertexDegrees_s[vertex]!=compStartingVertexDegree) {\n                                    not_Clique_Cycle = 1;\n                                }\n                                break;\n                            }\n                        }\n                    }\n                }\n            }\n            __syncthreads();\n        } while (found_new_level);\n        \n        if (!not_Clique_Cycle) {\n            if (compStartingVertexDegree == 2) {\n                *numDeletedVertices += (compSize+1)/2;\n                if (threadIdx.x == 0) {\n                    stackNumDeletedVertices[parentIndex] += (compSize+1)/2;\n                    --num_components;\n                }\n            } else if (compStartingVertexDegree == compSize-1) {\n                *numDeletedVertices += compSize-1;\n                if (threadIdx.x == 0) {\n                    stackNumDeletedVertices[parentIndex] += compSize-1;\n                    --num_components;\n                }\n            } else {\n                 __syncthreads();\n                if (threadIdx.x == 0) {\n                    not_Clique_Cycle = 1;\n                }\n            }\n        }\n        __syncthreads();\n\n        for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n            if(components[vertex] == 0) {\n                done = false;\n                break;\n            }\n        }\n        __syncthreads();\n        \n        if (!not_Clique_Cycle) {\n            for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n                if(components[vertex] > 0){\n                    components[vertex] = -1;\n                    vertexDegrees_s[vertex] = -1;\n                }\n            }\n            __syncthreads();\n                \n            if (!done) {\n                continue;\n            }\n            __syncthreads();\n\n            if (num_components==0) {\n                --(*stackTop);\n                return 0;\n            } else if (num_components==1) {\n                (*stackTop) -= 2;\n                return num_components;\n            }\n            __syncthreads();\n\n            break;\n        }\n        __syncthreads();\n        \n        if(done && num_components == 1) {\n            --(*stackTop);\n            return num_components;\n        }\n        \n        for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n            if(components[vertex] == 1) {\n                components[vertex] = vertexDegrees_s[vertex];\n            }\n        }\n        __syncthreads();\n         \n        unsigned int numDeletedVertices2 = 0;\n        startTime(PUSH_TO_STACK, blockCounters);\n        pushStack(vertexNum, (INT_OR_SHORT*)components, &numDeletedVertices2, parentIndex, 1, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, stackCompNode, stackTop, true);\n        endTime(PUSH_TO_STACK,blockCounters);\n        maxDepth(*stackTop, blockCounters);\n        __syncthreads();\n        \n        if (threadIdx.x==0) {\n            if (*stackTop==stackSize-1) {\n                exceedFlag = true;\n            }\n            \n        }\n        __syncthreads();\n        \n        if (exceedFlag) {\n            *stackTop = parentIndex-1;\n            *numDeletedVertices = minimum_s;\n            return 0; //no need for this node\n        }\n\n        for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x){\n            if(components[vertex] > 0){\n                components[vertex] = -1;\n            }\n        }\n        __syncthreads();\n\n    } while(!done);\n    __syncthreads();\n  \n    if (threadIdx.x==0) {  \n        for (unsigned int i=1; i<=num_components; ++i) {\n            stackNumDeletedVertices[parentIndex] += (int)minimum_s-(int)*numDeletedVertices-i+1;\n        }\n        done = false;\n        #if USE_SHORT_DEGREE\n        saveIntInShortArray(&stackVertexDegrees[(long long)parentIndex*(long long)graph.vertexNum+(long long)3], (int)minimum_s-(int)*numDeletedVertices-num_components+1);\n        #else\n        stackVertexDegrees[(long long)parentIndex*(long long)graph.vertexNum+(long long)3] = (int)minimum_s-(int)*numDeletedVertices-num_components+1;\n        #endif\n        if (minimum_s>*numDeletedVertices+num_components) {\n            #if USE_SHORT_DEGREE\n            saveIntInShortArray(&stackVertexDegrees[(long long)parentIndex*(long long)graph.vertexNum], minimum_s-*numDeletedVertices-num_components+1);\n            #else\n            stackVertexDegrees[(long long)parentIndex*(long long)graph.vertexNum] = minimum_s-*numDeletedVertices-num_components+1;\n            #endif\n        } else {\n            done = true;\n        }\n    }\n    __syncthreads();\n    \n    if (done) {\n        *stackTop = parentIndex-1;\n        *numDeletedVertices = minimum_s;\n        return 0; //no need for this node\n    }\n    __syncthreads();\n\n    return num_components;\n}\n\n__device__ unsigned int bfs_Hybrid(unsigned int vertexNum, INT_OR_SHORT* vertexDegrees_s, CSRGraph graph, INT_OR_SHORT* shared_mem, int arrayIndex, unsigned int minimum, unsigned int numDeletedVertices, CompArray compArray, WorkList<INT_OR_SHORT> workList, Counters* blockCounters, volatile INT_OR_SHORT * stackVertexDegrees, volatile unsigned int * stackNumDeletedVertices, volatile int * stackArrayIndex, int* stackTop, unsigned int stackSize, unsigned int compArraySize, unsigned int left, unsigned int right) {\n    __shared__ int globIndex;\n    __shared__ int num_components;\n    __shared__ bool exceedFlag;\n    __shared__ bool done;\n    __shared__ int found_new_level;\n    __shared__ int select_comp;\n    __shared__ int parentArrayIndex;\n    __shared__ int childArrayIndex;\n    __shared__ int nextChildIndex;\n    __shared__ int compSize;\n\n    volatile INT_OR_SHORT* components = shared_mem;    \n    unsigned int start = threadIdx.x + left;\n    \n    if (threadIdx.x == 0) {\n        done = true;\n    }\n    __syncthreads();\n\n    for(unsigned int vertex = threadIdx.x; vertex < graph.vertexNum; vertex+=blockDim.x) {\n        if(vertexDegrees_s[vertex] > 0) {\n            components[vertex] = 0;\n            done = false;\n        } else {\n            components[vertex] = -1;\n        }\n    }\n    __syncthreads();\n\n    if (done) {return 0;}\n    \n    if (threadIdx.x == 0) {\n        globIndex = atomicOr((int*)compArray.globalArrayIndex,0);\n        num_components = 0;\n        exceedFlag = false;\n    }\n    __syncthreads();\n\n    if (globIndex+3>compArraySize) {return 1;}\n\n    do {\n        __syncthreads();\n        if(threadIdx.x == 0) {\n            select_comp = 0;\n            ++num_components;\n            compSize = 1;\n            done = true;\n        }\n        __syncthreads();\n\n        for (unsigned int i = start; i<right+1 && !select_comp; i+=blockDim.x) {\n            if(components[i] == 0) {\n                if(atomicCAS(&select_comp,0,1) == 0) {\n                    components[i] = 1;                   \n                }\n            }\n        }\n        __syncthreads();\n\n        do {\n            __syncthreads();\n            if (threadIdx.x == 0) {\n                found_new_level = 0;\n            }\n             __syncthreads();\n            for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n                if(components[vertex] == 0) {\n                    for(unsigned int edge = graph.srcPtr[vertex]; edge < graph.srcPtr[vertex+1]; ++edge) {\n                        unsigned int neigh = graph.dst[edge];\n                        if (vertexDegrees_s[neigh]>0) {\n                            if(components[neigh] == 1) {\n                                components[vertex] = 1;\n                                atomicAdd(&compSize,1);\n                                found_new_level = 1;\n                                break;\n                            }\n                        }\n                    }\n                }\n            }\n            __syncthreads();\n        } while (found_new_level);\n\n        for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n            if(components[vertex] == 0) {\n                done = false;\n                break;\n            }\n        }\n        __syncthreads();\n        \n        if (num_components == 1) {\n            if (done) {break;}\n            __syncthreads();\n            if(threadIdx.x == 0) {\n                parentArrayIndex = atomicAdd((int*)compArray.globalArrayIndex,3);\n                if (parentArrayIndex<compArraySize-2) {\n                    compArray.entries[parentArrayIndex].index = arrayIndex;\n                    compArray.entries[parentArrayIndex].minimum = numDeletedVertices;\n                    compArray.entries[parentArrayIndex].counter = 1;\n                    childArrayIndex = parentArrayIndex+1;\n                    nextChildIndex = parentArrayIndex+2;\n                } else {\n                    done = true;\n                }\n            }\n            __syncthreads();\n            if (done) {break;}\n        } else {\n            if (threadIdx.x == 0) {\n                childArrayIndex = nextChildIndex;\n                nextChildIndex = atomicAdd((int*)compArray.globalArrayIndex,1);\n                if (nextChildIndex>=compArraySize) {\n                    done = true;\n                }\n            }\n        }\n        __syncthreads();\n        \n        if (threadIdx.x==0) {\n            if (!done) {\n                atomicAdd((int*)&compArray.entries[parentArrayIndex].counter, 1);\n            } else {\n                compSize = 0;\n            }\n        }\n        __syncthreads();\n\n        if (done) {\n            for(unsigned int vertex = start; vertex<right+1; vertex+=blockDim.x) {\n                if(components[vertex] == 1 || components[vertex] == 0) {\n                    components[vertex] = vertexDegrees_s[vertex];\n                    atomicAdd(&compSize,1);\n                }\n            }\n        } else {\n            for(unsigned int vertex = start; vertex<right+1; vertex+=blockDim.x) {\n                if(components[vertex] == 1) {\n                    components[vertex] = vertexDegrees_s[vertex];\n                }\n            }\n        }\n        __syncthreads();\n\n        if(threadIdx.x == 0) {\n            compArray.entries[childArrayIndex].index = parentArrayIndex;\n            compArray.entries[childArrayIndex].counter = 1;\n            if (minimum-numDeletedVertices>num_components) {\n                compArray.entries[childArrayIndex].minimum = (compSize-1<minimum-numDeletedVertices-num_components+1)?compSize-1:minimum-numDeletedVertices-num_components+1;\n            } else {\n                compArray.entries[childArrayIndex].minimum = 0;\n                atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum, minimum);\n                if (!done) {\n                    atomicSub((int*)&compArray.entries[parentArrayIndex].counter, 1);\n                }\n                done = true;\n                ++num_components;\n            }\n        }\n        __syncthreads();\n \n        bool enqueueSuccess;\n        unsigned int numDeletedVertices2 = 0;\n        if(checkThreshold(workList)) {\n            startTime(ENQUEUE,blockCounters);\n            enqueueSuccess = enqueue((INT_OR_SHORT*)components, workList, vertexNum, &numDeletedVertices2, childArrayIndex);\n        } else  {\n            enqueueSuccess = false;\n        }\n        __syncthreads();\n\n        if(!enqueueSuccess) {\n            startTime(PUSH_TO_STACK,blockCounters);\n            pushStack(vertexNum, (INT_OR_SHORT*)components, &numDeletedVertices2, childArrayIndex, NULL, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, NULL, stackTop, false);        \n            endTime(PUSH_TO_STACK,blockCounters);\n            maxDepth(*stackTop, blockCounters);\n            if (threadIdx.x==0 && *stackTop==stackSize-1) {\n                exceedFlag = true;\n                compArray.entries[childArrayIndex].minimum = 0;\n                atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum, minimum);\n                if (!done) {\n                    atomicSub((int*)&compArray.entries[parentArrayIndex].counter, 1);\n                }\n                done = true;\n                ++num_components;\n            }\n            __syncthreads();\n            if (exceedFlag) {--(*stackTop);}\n        } else {\n            endTime(ENQUEUE,blockCounters);\n        }\n        __syncthreads();\n\n        if (!done) {\n            for(unsigned int vertex = threadIdx.x; vertex < graph.vertexNum; vertex+=blockDim.x) {\n                if(components[vertex] > 0){\n                    components[vertex] = -1;\n                }\n            }\n        }\n        __syncthreads();\n\n    } while(!done);\n\n    return num_components;\n}\n\n__device__ unsigned int bfsCC_Hybrid(unsigned int vertexNum, INT_OR_SHORT* vertexDegrees_s, CSRGraph graph, INT_OR_SHORT* shared_mem, int arrayIndex, unsigned int minimum, unsigned int * numDeletedVertices, CompArray compArray, WorkList<INT_OR_SHORT> workList, Counters* blockCounters, volatile INT_OR_SHORT * stackVertexDegrees, volatile unsigned int * stackNumDeletedVertices, volatile int * stackArrayIndex, int* stackTop, unsigned int stackSize, unsigned int compArraySize, unsigned int left, unsigned int right, unsigned int* minimum_original) {\n    __shared__ int globIndex;\n    __shared__ int num_components;\n    __shared__ bool exceedFlag;\n    __shared__ bool done;\n    __shared__ int found_new_level;\n    __shared__ int select_comp;\n    __shared__ int parentArrayIndex;\n    __shared__ int childArrayIndex;\n    __shared__ int nextChildIndex;\n    __shared__ int compSize;\n    __shared__ int not_Clique_Cycle;\n    __shared__ int compStartingVertexDegree;\n    __shared__ int s_degree_sum_i;\n    __shared__ int s_max_deg_i;\n    __shared__ int s_fast_path_type;\n    __shared__ int s_tree_cover;\n\n    volatile INT_OR_SHORT* components = shared_mem;    \n    unsigned int start = threadIdx.x + left;\n    \n    if (threadIdx.x == 0) {\n        done = true;\n    }\n    __syncthreads();\n\n    for(unsigned int vertex = threadIdx.x; vertex < graph.vertexNum; vertex+=blockDim.x) {\n        if(vertexDegrees_s[vertex] > 0) {\n            components[vertex] = 0;\n            done = false;\n        } else {\n            components[vertex] = -1;\n        }\n    }\n    __syncthreads();\n\n    if (done) {return 0;}\n    \n    if (threadIdx.x == 0) {\n        globIndex = atomicOr((int*)compArray.globalArrayIndex,0);\n        num_components = 0;\n        exceedFlag = false;\n    }\n    __syncthreads();\n\n    if (globIndex+3>compArraySize) {return 1;}\n\n    do {\n        __syncthreads();\n        if(threadIdx.x == 0) {\n            select_comp = 0;\n            not_Clique_Cycle = 0;\n            ++num_components;\n            compSize = 1;\n            done = true;\n        }\n        __syncthreads();\n\n        for (unsigned int i = start; i<right+1 && !select_comp; i+=blockDim.x) {\n            if(components[i] == 0) {\n                if(atomicCAS(&select_comp,0,1) == 0) {\n                    components[i] = 1;\n                    compStartingVertexDegree = vertexDegrees_s[i];\n                }\n            }\n        }\n        __syncthreads();\n\n        do {\n            __syncthreads();\n            if (threadIdx.x == 0) {\n                found_new_level = 0;\n            }\n             __syncthreads();\n            for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n                if(components[vertex] == 0) {\n                    for(unsigned int edge = graph.srcPtr[vertex]; edge < graph.srcPtr[vertex+1]; ++edge) {\n                        unsigned int neigh = graph.dst[edge];\n                        if (vertexDegrees_s[neigh]>0) {\n                            if(components[neigh] == 1) {\n                                components[vertex] = 1;\n                                atomicAdd(&compSize,1);\n                                found_new_level = 1;\n                                if (vertexDegrees_s[vertex]!=compStartingVertexDegree) {\n                                    not_Clique_Cycle = 1;\n                                }\n                                break;\n                            }\n                        }\n                    }\n                }\n            }\n            __syncthreads();\n        } while (found_new_level);\n\n        if (threadIdx.x == 0) {\n            s_degree_sum_i = 0;\n            s_max_deg_i = 0;\n            s_fast_path_type = 0;\n            s_tree_cover = 0;\n        }\n        __syncthreads();\n\n        int local_deg_sum = 0;\n        int local_max_deg = 0;\n        for (unsigned int v = start; v < right + 1; v += blockDim.x) {\n            if (components[v] == 1 && vertexDegrees_s[v] > 0) {\n                int d = vertexDegrees_s[v];\n                local_deg_sum += d;\n                if (d > local_max_deg) local_max_deg = d;\n            }\n        }\n        atomicAdd(&s_degree_sum_i, local_deg_sum);\n        atomicMax(&s_max_deg_i, local_max_deg);\n        __syncthreads();\n\n        if (threadIdx.x == 0) {\n            int n_i = compSize;\n            int m_i = s_degree_sum_i / 2; // Undirected edge count\n            if (n_i >= 3 && m_i == n_i && s_max_deg_i == 2) {\n                s_fast_path_type = 1; // Simple Cycle\n            } else if (n_i >= 1 && m_i == n_i - 1) {\n                s_fast_path_type = 2; // Acyclic Tree / Star\n            } else if (n_i >= 2 && compStartingVertexDegree == n_i - 1 && !not_Clique_Cycle) {\n                s_fast_path_type = 3; // Clique\n            }\n        }\n        __syncthreads();\n\n        if (s_fast_path_type == 1) {\n            int cycle_cover = (compSize + 1) / 2;\n            *numDeletedVertices += cycle_cover;\n            if (threadIdx.x == 0) {\n                #if USE_COUNTERS\n                blockCounters->totalTime[CYCLE_FAST_PATH_HITS]++;\n                #endif\n                if (num_components > 1) {\n                    atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum, cycle_cover);\n                }\n                --num_components;\n            }\n        } else if (s_fast_path_type == 2) {\n            if (compSize == 1) {\n                if (threadIdx.x == 0) {\n                    --num_components;\n                }\n            } else if (compSize == 2 || s_max_deg_i == compSize - 1) {\n                *numDeletedVertices += 1;\n                if (threadIdx.x == 0) {\n                    #if USE_COUNTERS\n                    blockCounters->totalTime[TREE_FAST_PATH_HITS]++;\n                    #endif\n                    if (num_components > 1) {\n                        atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum, 1);\n                    }\n                    --num_components;\n                }\n            } else {\n                __shared__ int selected_leaf;\n                __shared__ int parent_v;\n                while (true) {\n                    if (threadIdx.x == 0) {\n                        selected_leaf = -1;\n                        parent_v = -1;\n                    }\n                    __syncthreads();\n\n                    for (unsigned int v = start; v < right + 1; v += blockDim.x) {\n                        if (components[v] == 1 && vertexDegrees_s[v] == 1) {\n                            atomicMax(&selected_leaf, (int)v);\n                            break;\n                        }\n                    }\n                    __syncthreads();\n\n                    if (selected_leaf == -1) break;\n\n                    int u = selected_leaf;\n                    for (unsigned int e = graph.srcPtr[u] + threadIdx.x; e < graph.srcPtr[u + 1]; e += blockDim.x) {\n                        unsigned int nb = graph.dst[e];\n                        if (components[nb] == 1 && vertexDegrees_s[nb] > 0) {\n                            parent_v = (int)nb;\n                        }\n                    }\n                    __syncthreads();\n\n                    if (parent_v != -1) {\n                        int p = parent_v;\n                        for (unsigned int e = graph.srcPtr[p] + threadIdx.x; e < graph.srcPtr[p + 1]; e += blockDim.x) {\n                            unsigned int nb = graph.dst[e];\n                            if (components[nb] == 1 && vertexDegrees_s[nb] > 0) {\n                                atomicSub((int*)&vertexDegrees_s[nb], 1);\n                            }\n                        }\n                        __syncthreads();\n                        if (threadIdx.x == 0) {\n                            vertexDegrees_s[p] = 0;\n                            s_tree_cover += 1;\n                        }\n                        __syncthreads();\n                    } else {\n                        if (threadIdx.x == 0) vertexDegrees_s[u] = 0;\n                        __syncthreads();\n                    }\n                }\n\n                *numDeletedVertices += s_tree_cover;\n                if (threadIdx.x == 0) {\n                    #if USE_COUNTERS\n                    blockCounters->totalTime[TREE_FAST_PATH_HITS]++;\n                    #endif\n                    if (num_components > 1) {\n                        atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum, s_tree_cover);\n                    }\n                    --num_components;\n                }\n            }\n        } else if (s_fast_path_type == 3) {\n            int clique_cover = compSize - 1;\n            *numDeletedVertices += clique_cover;\n            if (threadIdx.x == 0) {\n                if (num_components > 1) {\n                    atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum, clique_cover);\n                }\n                --num_components;\n            }\n        }\n        __syncthreads();\n\n        for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n            if(components[vertex] == 0) {\n                done = false;\n                break;\n            }\n        }\n        __syncthreads();\n        \n        if (s_fast_path_type > 0) {\n            if (!done) {\n                for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n                    if(components[vertex] > 0){\n                        components[vertex] = -1;\n                        vertexDegrees_s[vertex] = -1;\n                    }\n                }\n                __syncthreads();\n                continue;\n            }\n\n            if (num_components==0) {return 0;} \n\n            if (threadIdx.x == 0) {\n                arrayIndex = parentArrayIndex;\n                int numDelVer;\n                int counter = atomicSub((int*)&compArray.entries[arrayIndex].counter, 1);\n                if (counter==1) {\n                    int parentIndex = compArray.entries[arrayIndex].index;\n                    atomicAdd((int*)&compArray.entries[parentIndex].minimum, compArray.entries[arrayIndex].minimum);\n                    int parentCounter = atomicSub((int*)&compArray.entries[parentIndex].counter, 1);\n                    if (parentCounter==1) {\n                        arrayIndex = compArray.entries[parentIndex].index;\n                        numDelVer = atomicOr((int*)&compArray.entries[parentIndex].minimum,0);\n                        while (true) {\n                            if (arrayIndex>-1) {\n                                atomicMin((int*)&compArray.entries[arrayIndex].minimum, numDelVer);\n                                int counter = atomicSub((int*)&compArray.entries[arrayIndex].counter, 1);\n                                if (counter-1>0) {\n                                    break;\n                                }\n                                int parentIndex = compArray.entries[arrayIndex].index;\n                                atomicAdd((int*)&compArray.entries[parentIndex].minimum, compArray.entries[arrayIndex].minimum);\n                                int parentCounter = atomicSub((int*)&compArray.entries[parentIndex].counter, 1);\n                                if (parentCounter-1>0) {\n                                    break;\n                                }\n                                arrayIndex = compArray.entries[parentIndex].index;\n                                numDelVer = atomicOr((int*)&compArray.entries[parentIndex].minimum,0);\n\n                            } else {\n                                atomicMin(minimum_original, numDelVer);\n                                break;\n                            }\n                        }\n                    }\n                }\n            }\n            __syncthreads();\n            return num_components+1; //although num_components maybe 1, it will run on the compArray\n        }\n        __syncthreads();\n\n        if (num_components == 1) {\n            if (done) {break;}\n            __syncthreads();\n            if(threadIdx.x == 0) {\n                parentArrayIndex = atomicAdd((int*)compArray.globalArrayIndex,3);\n                if (parentArrayIndex<compArraySize-2) {\n                    compArray.entries[parentArrayIndex].index = arrayIndex;\n                    compArray.entries[parentArrayIndex].minimum = *numDeletedVertices;\n                    compArray.entries[parentArrayIndex].counter = 1;\n                    childArrayIndex = parentArrayIndex+1;\n                    nextChildIndex = parentArrayIndex+2;\n                } else {\n                    done = true;\n                }\n            }\n            __syncthreads();\n            if (done) {break;}\n        } else {\n            if (threadIdx.x == 0) {\n                childArrayIndex = nextChildIndex;\n                nextChildIndex = atomicAdd((int*)compArray.globalArrayIndex,1);\n                if (nextChildIndex>=compArraySize) {\n                    done = true;\n                }\n            }\n        }\n        __syncthreads();\n        \n        if (threadIdx.x==0) {\n            if (!done) {\n                atomicAdd((int*)&compArray.entries[parentArrayIndex].counter, 1);\n            } else {\n                compSize = 0;\n            }\n        }\n        __syncthreads();\n\n        if (done) {\n            for(unsigned int vertex = start; vertex<right+1; vertex+=blockDim.x) {\n                if(components[vertex] == 1 || components[vertex] == 0) {\n                    components[vertex] = vertexDegrees_s[vertex];\n                    atomicAdd(&compSize,1);\n                }\n            }\n        } else {\n            for(unsigned int vertex = start; vertex<right+1; vertex+=blockDim.x) {\n                if(components[vertex] == 1) {\n                    components[vertex] = vertexDegrees_s[vertex];\n                }\n            }\n        }\n        __syncthreads();\n\n        if(threadIdx.x == 0) {\n            compArray.entries[childArrayIndex].index = parentArrayIndex;\n            compArray.entries[childArrayIndex].counter = 1;\n            if (minimum>*numDeletedVertices+num_components) {\n                compArray.entries[childArrayIndex].minimum = (compSize-1<minimum-*numDeletedVertices-num_components+1)?compSize-1:minimum-*numDeletedVertices-num_components+1;\n            } else {\n                compArray.entries[childArrayIndex].minimum = 0;\n                atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum, minimum);\n                if (!done) {\n                    atomicSub((int*)&compArray.entries[parentArrayIndex].counter, 1);\n                }\n                done = true;\n                ++num_components;\n            }\n        }\n        __syncthreads();\n\n        bool enqueueSuccess;\n        unsigned int numDeletedVertices2 = 0;\n        if(checkThreshold(workList)) {\n            startTime(ENQUEUE,blockCounters);\n            enqueueSuccess = enqueue((INT_OR_SHORT*)components, workList, vertexNum, &numDeletedVertices2, childArrayIndex);\n        } else  {\n            enqueueSuccess = false;\n        }\n        __syncthreads();\n\n        if(!enqueueSuccess) {\n            startTime(PUSH_TO_STACK,blockCounters);\n            pushStack(vertexNum, (INT_OR_SHORT*)components, &numDeletedVertices2, childArrayIndex, NULL, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, NULL, stackTop, false);        \n            endTime(PUSH_TO_STACK,blockCounters);\n            maxDepth(*stackTop, blockCounters);\n            if (threadIdx.x==0 && *stackTop==stackSize-1) {\n                exceedFlag = true;\n                compArray.entries[childArrayIndex].minimum = 0;\n                atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum, minimum);\n                if (!done) {\n                    atomicSub((int*)&compArray.entries[parentArrayIndex].counter, 1);\n                }\n                done = true;\n                ++num_components;\n            }\n            __syncthreads();\n            if (exceedFlag) {--(*stackTop);}\n        } else {\n            endTime(ENQUEUE,blockCounters);\n        }\n        __syncthreads();\n\n        if (!done) {\n            for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n                if(components[vertex] > 0){\n                    components[vertex] = -1;\n                }\n            }\n        }\n        __syncthreads();\n\n    } while(!done);\n    \n    __syncthreads();\n    \n    return num_components;\n}\n\n__device__ unsigned int bfsConnectedComp_Hybrid(unsigned int vertexNum, INT_OR_SHORT* vertexDegrees_s, CSRGraph graph, INT_OR_SHORT* shared_mem, int arrayIndex, unsigned int minimum, unsigned int numDeletedVertices, CompArray compArray, WorkList<INT_OR_SHORT> workList, Counters* blockCounters, volatile INT_OR_SHORT * stackVertexDegrees, volatile unsigned int * stackNumDeletedVertices, volatile int * stackArrayIndex, int* stackTop, unsigned int stackSize, unsigned int compArraySize, unsigned int left, unsigned int right) {\n\n    __shared__ int globIndex;\n    __shared__ int num_components;\n    __shared__ bool exceedFlag;\n    __shared__ int select_comp1;\n    __shared__ int select_comp2;\n    __shared__ int comp1;\n    __shared__ int comp2;\n    __shared__ bool done;\n    __shared__ int found_new_level;\n    __shared__ int parentArrayIndex;\n    __shared__ int childArrayIndex;\n    __shared__ int nextChildIndex;\n    __shared__ int compSize;\n\n    volatile INT_OR_SHORT* components = shared_mem;\n    unsigned int start = threadIdx.x + left;\n    \n    if (threadIdx.x == 0) {\n        done = true;\n    }\n    __syncthreads();\n    \n    for(unsigned int vertex = threadIdx.x; vertex < graph.vertexNum; vertex+=blockDim.x) {\n        if(vertexDegrees_s[vertex] > 0) {\n            components[vertex] = vertex;\n            done = false;\n        } else {\n            components[vertex] = -1;\n        }\n    }\n    __syncthreads();\n\n    if (done) {return 0;}\n    \n    if (threadIdx.x == 0) {\n        globIndex = atomicOr((int*)compArray.globalArrayIndex,0);\n        num_components = 0;\n        select_comp1 = 0;\n        select_comp2 = 0;\n        exceedFlag = false;\n    }\n    __syncthreads();\n\n    if (globIndex+3>compArraySize) {\n        return 1;\n    }\n    __syncthreads();\n\n    do {\n        __syncthreads();\n        if (threadIdx.x == 0) {\n            found_new_level = 0;\n        }\n         __syncthreads();\n        for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n            if(components[vertex]>-1) {\n                for(unsigned int edge = graph.srcPtr[vertex]; edge < graph.srcPtr[vertex+1]; ++edge) {\n                    unsigned int neigh = graph.dst[edge];\n                    if (vertexDegrees_s[neigh]>0) {\n                        if(components[neigh] < components[vertex]) {\n                            components[vertex] = components[neigh];\n                            found_new_level = 1;\n                        }\n                    }\n                }\n            }\n        }\n        __syncthreads();\n    } while (found_new_level);\n        \n    for (unsigned int i = start; i < right+1 && !select_comp1; i+=blockDim.x) {\n        if(components[i]>-1) {\n            if(atomicCAS(&select_comp1,0,1) == 0) {\n                comp1 = components[i];                   \n            }\n        }\n    }\n    __syncthreads();\n\n    for (unsigned int i = start; i < right+1 && !select_comp2; i+=blockDim.x) {\n        if(components[i]>-1 && components[i] != comp1) {\n            if(atomicCAS(&select_comp2,0,1) == 0) {\n                comp2 = components[i];                   \n            }\n        }\n    }\n    __syncthreads();\n\n    if (threadIdx.x==0 && !select_comp2) {\n        done = true;\n    }\n    __syncthreads();\n\n    if (done) {return 1;}\n    __syncthreads();\n\n    if(threadIdx.x == 0) {\n        parentArrayIndex = atomicAdd((int*)compArray.globalArrayIndex,3);\n        if (parentArrayIndex<compArraySize-2) {\n            compArray.entries[parentArrayIndex].index = arrayIndex;\n            compArray.entries[parentArrayIndex].minimum = numDeletedVertices;\n            compArray.entries[parentArrayIndex].counter = 1;\n            childArrayIndex = parentArrayIndex+1;\n            nextChildIndex = parentArrayIndex+2;\n        } else {\n            done = true;\n        }\n    }\n    __syncthreads();\n\n    if (done) {return 1;}\n    __syncthreads();\n\n    while (true) {\n        __syncthreads();\n        if (threadIdx.x==0) {\n            select_comp2 = 0;\n            ++num_components;\n            compSize = 0;\n            if (!done) {\n                atomicAdd((int*)&compArray.entries[parentArrayIndex].counter, 1);\n            }\n\n            compArray.entries[childArrayIndex].index = parentArrayIndex;\n            compArray.entries[childArrayIndex].counter = 1;\n            if (minimum-numDeletedVertices>num_components) {\n                compArray.entries[childArrayIndex].minimum = minimum-numDeletedVertices-num_components+1;\n            } else {\n                compArray.entries[childArrayIndex].minimum = 0;\n                atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum, minimum);\n                if (!done) {\n                    atomicSub((int*)&compArray.entries[parentArrayIndex].counter, 1);\n                }\n                done = true;\n                ++num_components;\n            }\n        }\n        __syncthreads();\n        \n        bool enqueueSuccess;\n        unsigned int numDeletedVertices2 = 0;\n        if(checkThreshold(workList)) {\n            startTime(ENQUEUE,blockCounters);\n            enqueueSuccess = enqueue_connected_comp((INT_OR_SHORT*)vertexDegrees_s, (INT_OR_SHORT*)components, comp1, &compSize, done, workList, vertexNum, &numDeletedVertices2, childArrayIndex, compArray);\n        } else  {\n            enqueueSuccess = false;\n        }\n        __syncthreads();\n\n        if(!enqueueSuccess) {\n            startTime(PUSH_TO_STACK,blockCounters);\n            pushStack_connected_comp(vertexNum, (INT_OR_SHORT*)vertexDegrees_s, (INT_OR_SHORT*)components, comp1, &compSize, done, &numDeletedVertices2, childArrayIndex, NULL, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, NULL, stackTop, false, compArray); \n            endTime(PUSH_TO_STACK,blockCounters);\n            maxDepth(*stackTop, blockCounters);\n            if (threadIdx.x==0 && *stackTop==stackSize-1) {\n                exceedFlag = true;\n                compArray.entries[childArrayIndex].minimum = 0;\n                atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum, minimum);\n                if (!done) {\n                    atomicSub((int*)&compArray.entries[parentArrayIndex].counter, 1);\n                }\n                done = true;\n                ++num_components;\n            }\n            __syncthreads();\n            if (exceedFlag) {--(*stackTop);}\n        } else {\n            endTime(ENQUEUE,blockCounters);\n        }\n        __syncthreads();\n        \n        if(done) {break;}\n        __syncthreads();\n        \n        if (threadIdx.x==0) {\n            comp1 = comp2;\n        }\n        __syncthreads();\n\n        for (unsigned int i = start; i < right+1 && !select_comp2; i+=blockDim.x) {\n            if(components[i]>-1 && components[i] != comp1) {\n                if(atomicCAS(&select_comp2,0,1) == 0) {\n                    comp2 = components[i];                   \n                }\n            }\n        }\n        __syncthreads();\n        \n        if (threadIdx.x == 0) {\n            if (!select_comp2) {\n                done = true;\n            }\n            childArrayIndex = nextChildIndex;\n            nextChildIndex = atomicAdd((int*)compArray.globalArrayIndex,1);\n            if (nextChildIndex>=compArraySize) {\n                done = true;\n            }\n        }\n        __syncthreads();\n    }\n    return num_components;\n}\n\n__device__ unsigned int bfsCC_Param_Hybrid(unsigned int vertexNum, INT_OR_SHORT* vertexDegrees_s, CSRGraph graph, INT_OR_SHORT* shared_mem, int arrayIndex, unsigned int minimum, unsigned int* numDeletedVertices, CompArray compArray, WorkList<INT_OR_SHORT> workList, Counters* blockCounters, volatile INT_OR_SHORT * stackVertexDegrees, volatile unsigned int * stackNumDeletedVertices, volatile int * stackArrayIndex, int* stackTop, unsigned int stackSize, unsigned int compArraySize, unsigned int * k, unsigned int * kFound, unsigned int left, unsigned int right) {\n\n    __shared__ int globIndex;\n    __shared__ int num_components;\n    __shared__ bool exceedFlag;\n    __shared__ bool done;\n    __shared__ int found_new_level;\n    __shared__ int select_comp;\n    __shared__ int parentArrayIndex;\n    __shared__ int childArrayIndex;\n    __shared__ int nextChildIndex;\n    __shared__ int compSize;\n    __shared__ int not_Clique_Cycle;\n    __shared__ int compStartingVertexDegree;\n\n    volatile INT_OR_SHORT* components = shared_mem;    \n    unsigned int start = threadIdx.x + left;\n    \n    if (threadIdx.x == 0) {\n        done = true;\n    }\n    __syncthreads();\n\n    for(unsigned int vertex = threadIdx.x; vertex < graph.vertexNum; vertex+=blockDim.x) {\n        if(vertexDegrees_s[vertex] > 0) {\n            components[vertex] = 0;\n            done = false;\n        } else {\n            components[vertex] = -1;\n        }\n    }\n    __syncthreads();\n\n    if (done) {return 0;}\n    \n    if (threadIdx.x == 0) {\n        globIndex = atomicOr((int*)compArray.globalArrayIndex,0);\n        num_components = 0;\n        exceedFlag = false;\n    }\n    __syncthreads();\n\n    if (globIndex+3>compArraySize) {return 1;}\n\n    do {\n        __syncthreads();\n        if(threadIdx.x == 0) {\n            select_comp = 0;\n            not_Clique_Cycle = 0;\n            ++num_components;\n            compSize = 1;\n            done = true;\n        }\n        __syncthreads();\n\n        for (unsigned int i = start; i<right+1 && !select_comp; i+=blockDim.x) {\n            if(components[i] == 0) {\n                if(atomicCAS(&select_comp,0,1) == 0) {\n                    components[i] = 1;\n                    compStartingVertexDegree = vertexDegrees_s[i];\n                }\n            }\n        }\n        __syncthreads();\n\n        do {\n            __syncthreads();\n            if (threadIdx.x == 0) {\n                found_new_level = 0;\n            }\n             __syncthreads();\n            for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n                if(components[vertex] == 0) {\n                    for(unsigned int edge = graph.srcPtr[vertex]; edge < graph.srcPtr[vertex+1]; ++edge) {\n                        unsigned int neigh = graph.dst[edge];\n                        if (vertexDegrees_s[neigh]>0) {\n                            if(components[neigh] == 1) {\n                                components[vertex] = 1;\n                                atomicAdd(&compSize,1);\n                                found_new_level = 1;\n                                if (vertexDegrees_s[vertex]!=compStartingVertexDegree) {\n                                    not_Clique_Cycle = 1;\n                                }\n                                break;\n                            }\n                        }\n                    }\n                }\n            }\n            __syncthreads();\n        } while (found_new_level);\n\n        if (!not_Clique_Cycle) {\n            if (compStartingVertexDegree == 2) {\n                *numDeletedVertices += (compSize+1)/2;\n                if (threadIdx.x == 0) {\n                    if (num_components>1) {\n                        atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum,(compSize+1)/2); \n                    }\n                    --num_components;\n                }\n            } else if (compStartingVertexDegree == compSize-1) {\n                *numDeletedVertices += compSize-1;\n                if (threadIdx.x == 0) {\n                    if (num_components>1) {\n                        atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum,compSize-1); \n                    }\n                    --num_components;\n                }\n            } else {\n                 __syncthreads();\n                if (threadIdx.x == 0) {\n                    not_Clique_Cycle = 1;\n                }\n            }\n        }\n        __syncthreads();\n\n        for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n            if(components[vertex] == 0) {\n                done = false;\n                break;\n            }\n        }\n        __syncthreads();\n        \n        if (!not_Clique_Cycle) {\n            if (!done) {\n                for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n                    if(components[vertex] > 0){\n                        components[vertex] = -1;\n                        vertexDegrees_s[vertex] = -1;\n                    }\n                }\n                __syncthreads();\n                continue;\n            }\n\n            if (num_components==0) {return 0;}\n            \n            if (threadIdx.x == 0) {\n                int theArrayIndex = 0;\n                int numDelVer = 0;\n                unsigned int compMin = *k+1;\n                unsigned int iterationCounter = 0;\n                int parentIndex = parentArrayIndex;\n                while (true) {\n                    if (theArrayIndex>-1) {\n                        if (iterationCounter>0) {\n                            compMin = atomicMin((int*)&compArray.entries[theArrayIndex].minimum, numDelVer);\n                            if (compMin<=numDelVer) {\n                                break;\n                            }\n                            parentIndex = compArray.entries[theArrayIndex].index;\n                        }\n                        atomicSub((int*)&compArray.entries[parentIndex].minimum, compMin-numDelVer);\n                        theArrayIndex = compArray.entries[parentIndex].index;\n                        numDelVer = compArray.entries[parentIndex].minimum;\n                    } else {\n                        if(numDelVer<=*k) {\n                            atomicOr(kFound,1);\n                        }\n                        // For Debugging Purposes\n                        //printf("numDel = %d\\n",numDelVer);\n                        break;\n                    }\n                    ++iterationCounter;\n                }\n            }\n            __syncthreads();\n            return num_components+1; //although num_components maybe 1, it will run on the compArray\n        }\n        __syncthreads();\n        \n        if (num_components == 1) {\n            if (done) {break;}\n            __syncthreads();\n            if(threadIdx.x == 0) {\n                parentArrayIndex = atomicAdd((int*)compArray.globalArrayIndex,3);\n                if (parentArrayIndex<compArraySize-2) {\n                    compArray.entries[parentArrayIndex].index = arrayIndex;\n                    compArray.entries[parentArrayIndex].minimum = *numDeletedVertices+*k+1;\n                    childArrayIndex = parentArrayIndex+1;\n                    nextChildIndex = parentArrayIndex+2;\n                } else {\n                    done = true;\n                }\n            }\n            __syncthreads();\n            if (done) {break;}\n        } else {\n            if (threadIdx.x == 0) {\n                childArrayIndex = nextChildIndex;\n                nextChildIndex = atomicAdd((int*)compArray.globalArrayIndex,1);\n                if (nextChildIndex>=compArraySize) {\n                    done = true;\n                }\n            }\n        }\n        __syncthreads();\n        \n        if (done) {\n            for(unsigned int vertex = start; vertex<right+1; vertex+=blockDim.x) {\n                if(components[vertex] == 1 || components[vertex] == 0) {\n                    components[vertex] = vertexDegrees_s[vertex];\n                    atomicAdd(&compSize,1);\n                }\n            }\n        } else {\n            for(unsigned int vertex = start; vertex<right+1; vertex+=blockDim.x) {\n                if(components[vertex] == 1) {\n                    components[vertex] = vertexDegrees_s[vertex];\n                }\n            }\n        }\n        __syncthreads();\n\n        if(threadIdx.x == 0) {\n            compArray.entries[childArrayIndex].index = parentArrayIndex;\n            if (minimum>*numDeletedVertices+num_components) {\n                compArray.entries[childArrayIndex].minimum = (compSize-1<minimum-*numDeletedVertices-num_components+1)?compSize-1:minimum-*numDeletedVertices-num_components+1;\n                atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum,compArray.entries[childArrayIndex].minimum);\n            } else {\n                compArray.entries[childArrayIndex].minimum = 0;\n                atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum, *k+1);\n                done = true;\n                ++num_components;\n            }\n        }\n        __syncthreads();\n        \n        if (threadIdx.x==0) {\n            if (done) {\n                atomicSub((int*)&compArray.entries[parentArrayIndex].minimum, *k+1);\n            }\n        }\n        __syncthreads();\n \n        bool enqueueSuccess;\n        unsigned int numDeletedVertices2 = 0;\n        if(checkThreshold(workList)) {\n            startTime(ENQUEUE,blockCounters);\n            enqueueSuccess = enqueue((INT_OR_SHORT*)components, workList, vertexNum, &numDeletedVertices2, childArrayIndex);\n        } else  {\n            enqueueSuccess = false;\n        }\n        __syncthreads();\n\n        if(!enqueueSuccess) {\n            startTime(PUSH_TO_STACK,blockCounters);\n            pushStack(vertexNum, (INT_OR_SHORT*)components, &numDeletedVertices2, childArrayIndex, NULL, stackVertexDegrees, stackNumDeletedVertices, stackArrayIndex, NULL, stackTop, false);        \n            endTime(PUSH_TO_STACK,blockCounters);\n            maxDepth(*stackTop, blockCounters);\n            if (threadIdx.x==0 && *stackTop==stackSize-1) {\n                exceedFlag = true;\n                compArray.entries[childArrayIndex].minimum = 0;\n                atomicAdd((int*)&compArray.entries[parentArrayIndex].minimum, minimum);\n                if (!done) {\n                    atomicSub((int*)&compArray.entries[parentArrayIndex].counter, 1);\n                }\n                done = true;\n                ++num_components;\n            }\n            __syncthreads();\n            if (exceedFlag) {--(*stackTop);}\n        } else {\n            endTime(ENQUEUE,blockCounters);\n        }\n        __syncthreads();\n\n        if (!done) {\n            for(unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x) {\n                if(components[vertex] > 0){\n                    components[vertex] = -1;\n                }\n            }\n        }\n        __syncthreads();\n\n    } while(!done);\n \n    return num_components;\n}\n\n__device__ unsigned int highDegreeReductionRule(unsigned int vertexNum, INT_OR_SHORT *vertexDegrees_s, CSRGraph graph, INT_OR_SHORT * shared_mem, unsigned int numDeletedVertices, unsigned int minimum, unsigned int left, unsigned int right) {\n\n    __shared__ unsigned int numberDeleted_s;\n    __shared__ bool graphHasChanged;\n\n    volatile INT_OR_SHORT * markedForDeletion = shared_mem;\n    if (threadIdx.x==0){\n        numberDeleted_s = 0;\n    }\n    unsigned int start = threadIdx.x + left;\n    do{\n        volatile INT_OR_SHORT * vertexDegrees_v = vertexDegrees_s;\n        __syncthreads();\n        \n        for (unsigned int vertex = start ; vertex < right+1; vertex += blockDim.x){\n            markedForDeletion[vertex] = 0;\n        }\n        if (threadIdx.x==0){\n            graphHasChanged = false;\n        }\n        \n        __syncthreads();\n\n        for (unsigned int vertex = start ; vertex < right+1; vertex+=blockDim.x){\n            INT_OR_SHORT degree = vertexDegrees_v[vertex];\n            if (degree > 0 && degree + numDeletedVertices + numberDeleted_s >= minimum){\n                markedForDeletion[vertex]=1;\n            }\n        }\n        __syncthreads();\n        \n        for (unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x){\n            if(markedForDeletion[vertex]){ \n                graphHasChanged = true;\n                vertexDegrees_v[vertex] = -1;\n                atomicAdd(&numberDeleted_s,1);\n            }\n        }\n        \n        __syncthreads();\n                    \n        for (unsigned int vertex = start ; vertex < right+1; vertex+=blockDim.x){\n            if(markedForDeletion[vertex]){ \n                for(unsigned int edge = graph.srcPtr[vertex]; edge < graph.srcPtr[vertex + 1]; edge++){\n                    unsigned int neighbor = graph.dst[edge];\n                    if (vertexDegrees_v[neighbor] > 0) {\n                        ATOMIC_TYPE(atomicSub)(&vertexDegrees_s[neighbor],1);\n                    }\n                }\n            }\n        }\n    \n    }while(graphHasChanged);\n\n    __syncthreads();\n\n    return numberDeleted_s;\n}\n\n\n__device__ unsigned int triangleReductionRule(unsigned int vertexNum, INT_OR_SHORT *vertexDegrees_s, CSRGraph graph, INT_OR_SHORT* shared_mem, unsigned int left, unsigned int right) {\n    \n    __shared__ unsigned int numberDeleted;\n    __shared__ bool graphHasChanged;\n    \n    volatile INT_OR_SHORT * markedForDeletion = shared_mem;\n    if (threadIdx.x==0){\n        numberDeleted = 0;\n        //printf("triangle\\n");\n    }\n    unsigned int start = threadIdx.x + left;\n    do{\n        volatile INT_OR_SHORT * vertexDegrees_v = vertexDegrees_s;\n        __syncthreads();\n\n        for (unsigned int vertex = start ; vertex < right+1; vertex += blockDim.x){\n            markedForDeletion[vertex] = 0;\n        }\n\n        if (threadIdx.x==0){\n            graphHasChanged = false;\n        }\n        \n        __syncthreads();\n        \n        for (unsigned int vertex = start ; vertex < right+1; vertex+=blockDim.x){\n            \n            INT_OR_SHORT degree = vertexDegrees_v[vertex];\n            if (degree == 2){\n                unsigned int neighbor1, neighbor2;\n                bool foundNeighbor1 = false, keepNeighbors = false;\n                for(unsigned int edge = graph.srcPtr[vertex] ; edge < graph.srcPtr[vertex + 1]; ++edge) {\n                    unsigned int neighbor = graph.dst[edge];\n                    INT_OR_SHORT neighborDegree = vertexDegrees_v[neighbor];\n                    if(neighborDegree>0){\n                        if(neighborDegree == 1 || neighborDegree == 2 && neighbor < vertex){\n                            keepNeighbors = true;\n                            break;\n                        } else if(!foundNeighbor1){\n                            foundNeighbor1 = true;\n                            neighbor1 = neighbor;\n                        } else {\n                            neighbor2 = neighbor;    \n                            break;\n                        }\n                    }\n                }\n\n                if(!keepNeighbors) {\n                    bool found = binarySearch(graph.dst, graph.srcPtr[neighbor2], graph.srcPtr[neighbor2 + 1] - 1, neighbor1);\n                    if(found){\n                        // Triangle Found\n                        markedForDeletion[neighbor1] = true;\n                        markedForDeletion[neighbor2] = true;\n                        break;\n                    }\n                }\n            }\n        }\n        \n        __syncthreads();\n        \n        for (unsigned int vertex = start; vertex < right+1; vertex+=blockDim.x){\n            if(markedForDeletion[vertex]){ \n                graphHasChanged = true;\n                vertexDegrees_v[vertex] = -1;\n                atomicAdd(&numberDeleted,1);\n            }\n        }\n        \n        __syncthreads();\n        \n        for (unsigned int vertex = start ; vertex < right+1; vertex+=blockDim.x){\n            if(markedForDeletion[vertex]){ \n                for(unsigned int edge = graph.srcPtr[vertex]; edge < graph.srcPtr[vertex + 1]; edge++){\n                    unsigned int neighbor = graph.dst[edge];\n                    if (vertexDegrees_v[neighbor] > 0){\n                        ATOMIC_TYPE(atomicSub)(&vertexDegrees_s[neighbor],1);\n                    }\n                }\n            }\n        }\n    \n    }while(graphHasChanged);\n\n    __syncthreads();\n\n    return numberDeleted;\n}\n\n__device__ void findMaxDegree(unsigned int vertexNum, unsigned int *maxVertex, INT_OR_SHORT *maxDegree, INT_OR_SHORT *vertexDegrees_s, INT_OR_SHORT * shared_mem, unsigned int left, unsigned int right) {\n    *maxVertex = 0;\n    *maxDegree = 0;\n    unsigned int start = threadIdx.x + left;\n    for(unsigned int vertex = start; vertex < right+1; vertex += blockDim.x) {\n        INT_OR_SHORT degree = vertexDegrees_s[vertex];\n        if(degree > *maxDegree){ \n            *maxVertex = vertex;\n            *maxDegree = degree;\n        }\n    }\n\n    // Reduce max degree\n    INT_OR_SHORT * vertex_s = shared_mem;\n    INT_OR_SHORT * degree_s = &shared_mem[blockDim.x];\n    __syncthreads(); \n\n    #if USE_SHORT_DEGREE\n    vertex_s[threadIdx.x] = threadIdx.x;\n    #else\n    vertex_s[threadIdx.x] = *maxVertex;\n    #endif\n\n    degree_s[threadIdx.x] = *maxDegree;\n    __syncthreads();\n\n    for(unsigned int stride = blockDim.x/2; stride > 0; stride /= 2) {\n        if(threadIdx.x < stride) {\n            if(degree_s[threadIdx.x] < degree_s[threadIdx.x + stride]){\n                degree_s[threadIdx.x] = degree_s[threadIdx.x + stride];\n                vertex_s[threadIdx.x] = vertex_s[threadIdx.x + stride];\n            }\n        }\n        __syncthreads();\n    }\n    \n    *maxDegree = degree_s[0];\n    \n    #if USE_SHORT_DEGREE\n    __shared__ unsigned int tempMaxVertex;\n    if (threadIdx.x == vertex_s[0]) {\n        tempMaxVertex = *maxVertex;\n    }\n    __syncthreads();\n    *maxVertex = tempMaxVertex;\n    \n    #else\n    *maxVertex = vertex_s[0];\n    \n    #endif\n    __syncthreads();\n}\n\n__device__ unsigned int findNumOfEdges(unsigned int vertexNum, INT_OR_SHORT *vertexDegrees_s, INT_OR_SHORT * shared_mem, unsigned int left, unsigned int right) {\n\n    int sumDegree = 0;\n    unsigned int start = threadIdx.x + left;\n    for(unsigned int vertex = start; vertex < right+1; vertex += blockDim.x) {\n        int degree = vertexDegrees_s[vertex];\n        if(degree > 0){\n            sumDegree += degree;\n        }\n    }\n    __syncthreads();\n    INT_OR_SHORT* degree_s = shared_mem;\n    \n    #if USE_SHORT_DEGREE\n\n    saveIntInShortArray(&degree_s[2*threadIdx.x], sumDegree);\n    \n    __syncthreads();\n\n    for(unsigned int stride = blockDim.x; stride > 1; stride /= 2) {\n        if(2*threadIdx.x < stride) {\n            AddIntInShortArray(&degree_s[2*threadIdx.x],readIntFromShortArray(&degree_s[2*threadIdx.x+stride]));\n        }\n        __syncthreads();\n    }\n    return readIntFromShortArray(&degree_s[0])/2;\n\n    #else\n    \n    degree_s[threadIdx.x] = sumDegree;\n    \n    __syncthreads();\n    \n    for(unsigned int stride = blockDim.x/2; stride > 0; stride /= 2) {\n        if(threadIdx.x < stride) {\n            degree_s[threadIdx.x] += degree_s[threadIdx.x + stride];\n        }\n        __syncthreads();\n    }\n    return degree_s[0]/2;\n    \n    #endif\n}\n\n#undef INT_OR_SHORT\n#undef ATOMIC_TYPE(base)\n#endif')

print(">>> All 4 Solver Variants successfully deployed and ready for compilation! <<<")


In [ ]:
# 3. Two-Pass Compilation Protocol
# Mode 1: Timing Pass (USE_COUNTERS=0) for pure unperturbed kernel runtime
# Mode 2: Telemetry Pass (USE_COUNTERS=1) for hardware counters, nodes visited, and phase breakdowns

def configure_and_compile(repo_path, use_counters=0, arch="sm_75"):
    src_dir = os.path.join(repo_path, "src")
    makefile = os.path.join(src_dir, "Makefile")
    
    # Update Makefile architecture
    with open(makefile, "r") as f:
        mk = f.read()
    
    code_val = arch
    arch_val = "compute_" + arch.split("_")[1]
    mk_updated = mk.replace("sm_70", code_val).replace("compute_70", arch_val)
    
    with open(makefile, "w") as f:
        f.write(mk_updated)
        
    cmd = f"cd {src_dir} && make clean > /dev/null 2>&1 && make USE_COUNTERS={use_counters} -j$(nproc)"
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"Compilation FAILED for {repo_path}:\n{res.stderr}")
        return False
    return True

VARIANTS = {
    "Variant_A_2022": "/content/mvc_gpu_research/Parallel-Vertex-Cover-Algorithms-on-GPUs",
    "Variant_B_2026": "/content/mvc_gpu_research/component-aware-vertex-cover-gpu",
    "Variant_C_Lazy": "/content/mvc_gpu_research/solver_variant_c_lazy",
    "Variant_D_Full": "/content/mvc_gpu_research/solver_variant_d_full"
}

print("="*70)
print(f"COMPILING ALL 4 VARIANTS (Target: {compute_arch}, Timing Pass USE_COUNTERS=0)")
print("="*70)
for name, path in VARIANTS.items():
    print(f"Compiling {name} ...")
    ok = configure_and_compile(path, use_counters=0, arch=compute_arch)
    if ok:
        print(f"  [SUCCESS] {name} binary generated at: {path}/src/output")
    else:
        print(f"  [ERROR] Compilation failed for {name}")

print("="*70)
print("All binaries compiled successfully!")


In [ ]:
# 4. Multi-Instance Ablation Benchmark Runner
import time
import pandas as pd
import re

DATA_DIR = "/content/mvc_gpu_research/component-aware-vertex-cover-gpu/data"

# Select representative benchmarks: Category 1 (Sparse / Scale-Free) & Category 2 (Dense / Cliques)
BENCHMARKS = [
    "qc324.txt",
    "mhda416.txt",
    "c-fat500-5.txt",
    "US-power-grid.txt",
    "power-eris1176.txt",
    "LastFM-Asia.txt",
    "web-spam.txt",
    "web-webbase-2001.txt"
]

results = []
print("="*85)
print(f"{'Graph':<20} | {'Variant':<18} | {'MVC Size':<10} | {'Time (s)':<10} | {'Status'}")
print("="*85)

for graph_file in BENCHMARKS:
    graph_path = os.path.join(DATA_DIR, graph_file)
    if not os.path.exists(graph_path):
        print(f"Skipping {graph_file} (not found)")
        continue

    for v_name, v_path in VARIANTS.items():
        bin_path = os.path.join(v_path, "src", "output")
        cmd = f"{bin_path} -f {graph_path} -v HYBRID"
        
        t0 = time.time()
        res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        elapsed = time.time() - t0
        
        out = res.stdout
        # Extract MVC Size
        mvc_val = None
        m = re.search(r'Minimum is:\s*(\d+)', out)
        if m:
            mvc_val = int(m.group(1))
        else:
            m2 = re.findall(r'minimum\s*=\s*(\d+)', out, re.IGNORECASE)
            if m2:
                mvc_val = int(m2[-1])
            else:
                m3 = re.findall(r'(\d+)', out)
                mvc_val = int(m3[-1]) if m3 else -1

        status = "PASS" if res.returncode == 0 else "FAIL"
        results.append({
            "Graph": graph_file.replace(".txt", ""),
            "Variant": v_name,
            "MVC_Size": mvc_val,
            "Time_Sec": round(elapsed, 4),
            "Status": status
        })
        print(f"{graph_file.replace('.txt',''):<20} | {v_name:<18} | {str(mvc_val):<10} | {elapsed:<10.4f} | {status}")

df_results = pd.DataFrame(results)
df_results.to_csv("/content/mvc_gpu_research/results_ablation.csv", index=False)
print("="*85)
print("Benchmark sweep completed! Saved to /content/mvc_gpu_research/results_ablation.csv")


In [ ]:
# 5. Exactness Verification Pass against Ground-Truth MVCs.txt
GT_PATH = "/content/mvc_gpu_research/component-aware-vertex-cover-gpu/src/MVCs.txt"

ground_truths = {}
if os.path.exists(GT_PATH):
    with open(GT_PATH, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("-"):
                parts = line.split()
                if len(parts) >= 3:
                    ground_truths[parts[1]] = int(parts[2])

print("="*75)
print("EXACTNESS & OPTIMALITY VERIFICATION")
print("="*75)
all_passed = True
for idx, row in df_results.iterrows():
    g = row["Graph"]
    v = row["Variant"]
    val = row["MVC_Size"]
    expected = ground_truths.get(g)
    if expected is not None:
        if val == expected:
            print(f"  [PASS] {g:<20} | {v:<18} | MVC={val} == Expected={expected}")
        else:
            print(f"  [FAIL] {g:<20} | {v:<18} | MVC={val} != Expected={expected}")
            all_passed = False
    else:
        print(f"  [INFO] {g:<20} | {v:<18} | MVC={val} (Ground truth unrecorded)")

print("="*75)
if all_passed:
    print(">>> 100% MATHEMATICAL EXACTNESS CONFIRMED ACROSS ALL TEST CASES <<<")
else:
    print(">>> ATTENTION: Cover discrepancy found. Inspect logs! <<<")


In [ ]:
# 6. Hardware Telemetry Pass (USE_COUNTERS=1)
# Extracts:
# - Unproductive BFS scans wasted (%)
# - Nodes explored per SM
# - In-block fast-path hits (Cycle and Tree fast paths)

print("Recompiling Variant B, C, and D with USE_COUNTERS=1 for hardware profiling...")
for name in ["Variant_B_2026", "Variant_C_Lazy", "Variant_D_Full"]:
    configure_and_compile(VARIANTS[name], use_counters=1, arch=compute_arch)

telemetry_records = []
for graph_file in BENCHMARKS:
    graph_path = os.path.join(DATA_DIR, graph_file)
    g_name = graph_file.replace(".txt", "")
    
    for v_name in ["Variant_B_2026", "Variant_C_Lazy", "Variant_D_Full"]:
        bin_path = os.path.join(VARIANTS[v_name], "src", "output")
        res = subprocess.run(f"{bin_path} -f {graph_path} -v HYBRID", shell=True, capture_output=True, text=True)
        
        # Check generated Counters CSV
        cnt_file = os.path.join(VARIANTS[v_name], "src", "Counters", f"Counters_{graph_file}")
        total_bfs = 0
        unprod_bfs = 0
        cycle_hits = 0
        tree_hits = 0
        if os.path.exists(cnt_file):
            try:
                cdf = pd.read_csv(cnt_file)
                if "Total BFS Scans" in cdf.columns:
                    total_bfs = cdf["Total BFS Scans"].sum()
                if "Unproductive BFS Scans" in cdf.columns:
                    unprod_bfs = cdf["Unproductive BFS Scans"].sum()
                if "Cycle Fast-Path Hits" in cdf.columns:
                    cycle_hits = cdf["Cycle Fast-Path Hits"].sum()
                if "Tree Fast-Path Hits" in cdf.columns:
                    tree_hits = cdf["Tree Fast-Path Hits"].sum()
            except Exception as e:
                pass
                
        telemetry_records.append({
            "Graph": g_name,
            "Variant": v_name,
            "Total_BFS": total_bfs,
            "Unproductive_BFS": unprod_bfs,
            "Unproductive_Pct": round((unprod_bfs / total_bfs * 100), 2) if total_bfs > 0 else 0.0,
            "Cycle_FastPath": cycle_hits,
            "Tree_FastPath": tree_hits
        })

df_telemetry = pd.DataFrame(telemetry_records)
df_telemetry.to_csv("/content/mvc_gpu_research/summary_telemetry.csv", index=False)
print("Telemetry pass complete. Summary saved to /content/mvc_gpu_research/summary_telemetry.csv")
df_telemetry.head(10)


In [ ]:
# 7. Generate High-Resolution Publication Figures (300 DPI)
import matplotlib.pyplot as plt
import numpy as np

FIGURES_DIR = "/content/mvc_gpu_research/paper_figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.dpi': 300
})

pivot_time = df_results.pivot(index="Graph", columns="Variant", values="Time_Sec")
benchmarks = list(pivot_time.index)
x = np.arange(len(benchmarks))
width = 0.22

# Figure 1: Speedup vs 2026 Baseline
plt.figure(figsize=(11, 5.5))
speedup_c = pivot_time["Variant_B_2026"] / pivot_time["Variant_C_Lazy"]
speedup_d = pivot_time["Variant_B_2026"] / pivot_time["Variant_D_Full"]

plt.bar(x - width/2, speedup_c, width, label='Variant C (2026 + Lazy Trigger)', color='#3b82f6', edgecolor='black', alpha=0.85)
plt.bar(x + width/2, speedup_d, width, label='Variant D (Proposed Full Solver)', color='#10b981', edgecolor='black', alpha=0.9)
plt.axhline(1.0, color='#ef4444', linestyle='--', linewidth=1.5, label='2026 Baseline (1.0x)')

plt.ylabel('Runtime Speedup Factor (x)')
plt.title('End-to-End Speedup Relative to 2026 TPDS SOTA Baseline')
plt.xticks(x, benchmarks, rotation=25, ha='right')
plt.legend(frameon=True, loc='upper left')
plt.tight_layout()
fig1_p = os.path.join(FIGURES_DIR, "fig1_speedup_ablation.png")
plt.savefig(fig1_p)
plt.savefig(fig1_p.replace(".png", ".pdf"))
plt.show()

# Figure 2: Solve Time Comparison
plt.figure(figsize=(11, 5.5))
for col, col_name, marker, color in [
    ("Variant_A_2022", "Variant A (2022 Baseline)", "^", "#ef4444"),
    ("Variant_B_2026", "Variant B (2026 Baseline)", "s", "#f59e0b"),
    ("Variant_C_Lazy", "Variant C (2026 + Lazy)", "d", "#3b82f6"),
    ("Variant_D_Full", "Variant D (Proposed Solver)", "o", "#10b981")
]:
    if col in pivot_time.columns:
        plt.plot(benchmarks, pivot_time[col], marker=marker, linewidth=2, label=col_name, color=color)

plt.yscale('log')
plt.ylabel('Total Solve Time in Seconds (Log Scale)')
plt.title('Absolute Solve Time across Ablation Variants (Log Scale)')
plt.xticks(rotation=25, ha='right')
plt.legend(frameon=True)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
fig2_p = os.path.join(FIGURES_DIR, "fig2_solve_time_log.png")
plt.savefig(fig2_p)
plt.savefig(fig2_p.replace(".png", ".pdf"))
plt.show()

# Figure 3: Unproductive BFS Scans Reduction
if not df_telemetry.empty:
    pivot_bfs = df_telemetry.pivot(index="Graph", columns="Variant", values="Unproductive_Pct")
    plt.figure(figsize=(10, 5))
    if "Variant_B_2026" in pivot_bfs.columns and "Variant_D_Full" in pivot_bfs.columns:
        plt.bar(x - width/2, pivot_bfs["Variant_B_2026"], width, label='2026 Baseline Wasted Scans (%)', color='#f87171', edgecolor='black')
        plt.bar(x + width/2, pivot_bfs["Variant_D_Full"], width, label='Proposed Solver Wasted Scans (%)', color='#60a5fa', edgecolor='black')
        plt.ylabel('Unproductive BFS Scan Ratio (%)')
        plt.title('Elimination of Unproductive Parallel BFS Scans')
        plt.xticks(x, benchmarks, rotation=25, ha='right')
        plt.ylim(0, 100)
        plt.legend(frameon=True)
        plt.tight_layout()
        fig3_p = os.path.join(FIGURES_DIR, "fig3_bfs_overhead_reduction.png")
        plt.savefig(fig3_p)
        plt.savefig(fig3_p.replace(".png", ".pdf"))
        plt.show()

print(f"All publication figures exported to: {FIGURES_DIR}")


In [ ]:
# 8. Download Results & Publication Package
# Packages all generated figures (PNG & PDF), raw results CSV, and telemetry CSV into a zip file
from google.colab import files
import zipfile

OUTPUT_ZIP = "/content/Exact_MVC_GPU_Paper_Results.zip"
with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["/content/mvc_gpu_research/results_ablation.csv", "/content/mvc_gpu_research/summary_telemetry.csv"]:
        if os.path.exists(f):
            z.write(f, os.path.basename(f))
    for f in os.listdir(FIGURES_DIR):
        full_p = os.path.join(FIGURES_DIR, f)
        z.write(full_p, os.path.join("paper_figures", f))

print(f"Package created: {OUTPUT_ZIP} ({os.path.getsize(OUTPUT_ZIP)/1024:.1f} KB)")
print("Initiating automatic download to your browser...")
files.download(OUTPUT_ZIP)
